In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [1]:
import sys
print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python: {sys.version}")

packages = {
    'torch': 'torch', 'transformers': 'transformers',
    'sklearn': 'sklearn', 'lightgbm': 'lightgbm',
    'xgboost': 'xgboost', 'numpy': 'numpy',
    'pandas': 'pandas', 'scipy': 'scipy',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
    'tqdm': 'tqdm', 'joblib': 'joblib',
}

all_ok = True
for name, module in packages.items():
    try:
        mod = __import__(module)
        ver = getattr(mod, '__version__', 'unknown')
        print(f"  {name:<20} {ver:<15} ✅")
    except ImportError:
        print(f"  {name:<20} {'NOT FOUND':<15} ❌")
        all_ok = False

import torch
print(f"\n  CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA Version   : {torch.version.cuda}")
    print(f"  GPU            : {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"  GPU Memory     : {total/1024**3:.1f} GB")

print(f"\nStatus: {'✅ ALL OK' if all_ok else '❌ Fix missing packages'}")

ENVIRONMENT CHECK
Python: 3.10.8 | packaged by conda-forge | (main, Nov 24 2022, 14:07:00) [MSC v.1916 64 bit (AMD64)]
  torch                2.1.0+cu118     ✅


c:\Users\user\anaconda3\envs\ai_detector\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  transformers         4.40.0          ✅
  sklearn              1.4.2           ✅
  lightgbm             4.3.0           ✅
  xgboost              2.0.3           ✅
  numpy                1.26.4          ✅
  pandas               2.0.3           ✅
  scipy                1.11.4          ✅
  matplotlib           3.8.4           ✅
  seaborn              0.13.2          ✅
  tqdm                 4.66.4          ✅
  joblib               1.4.2           ✅

  CUDA Available : True
  CUDA Version   : 11.8
  GPU            : NVIDIA T1000 8GB
  GPU Memory     : 8.0 GB

Status: ✅ ALL OK


In [2]:
import os, sys, json, time, random, warnings, logging, gc
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy  as np
import pandas as pd
from scipy.stats import randint, uniform

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV,
    train_test_split, cross_validate,
)
from sklearn.preprocessing import (
    RobustScaler, MinMaxScaler, LabelEncoder
)
from sklearn.pipeline    import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model  import (
    LogisticRegression, RidgeClassifier, SGDClassifier
)
from sklearn.svm           import SVC, LinearSVC
from sklearn.ensemble      import (
    RandomForestClassifier, ExtraTreesClassifier
)
from sklearn.neighbors     import KNeighborsClassifier
from sklearn.naive_bayes   import MultinomialNB
from sklearn.tree          import DecisionTreeClassifier

import lightgbm as lgb
import xgboost  as xgb

import torch
import torch.nn            as nn
import torch.optim         as optim
import torch.nn.functional as F
from torch.utils.data      import Dataset, DataLoader
from torch.cuda.amp        import autocast, GradScaler

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import joblib
from tqdm.auto import tqdm

print("✅ All imports successful")

✅ All imports successful


In [3]:
# ── Seeds ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available()
                      else 'cpu')

# ── VRAM settings ─────────────────────────────────────────────
if torch.cuda.is_available():
    VRAM_GB = (torch.cuda.get_device_properties(0)
               .total_memory / 1024**3)
else:
    VRAM_GB = 0.0

if   VRAM_GB >= 16: BERT_BATCH=16; BERT_MAXLEN=512; GRAD_ACCUM=2
elif VRAM_GB >=  8: BERT_BATCH=8;  BERT_MAXLEN=256; GRAD_ACCUM=4
elif VRAM_GB >=  4: BERT_BATCH=4;  BERT_MAXLEN=128; GRAD_ACCUM=8
else:               BERT_BATCH=4;  BERT_MAXLEN=128; GRAD_ACCUM=8

USE_FP16        = torch.cuda.is_available()
DL_BATCH        = 64
BERT_LR         = 2e-5
DL_LR           = 1e-3
MAX_EPOCHS_BERT = 5
MAX_EPOCHS_DL   = 30
PATIENCE        = 4
N_FOLDS         = 5
TEST_SIZE       = 0.20
VAL_SIZE        = 0.20
N_ITER_SEARCH   = 20

# ── Paths ─────────────────────────────────────────────────────
DATA_PATH  = r'training.csv'

OUTPUT_DIR = Path('ml_results')
PLOTS_DIR  = OUTPUT_DIR / 'plots'
MODELS_DIR = OUTPUT_DIR / 'models'
TABLES_DIR = OUTPUT_DIR / 'tables'
for d in [OUTPUT_DIR, PLOTS_DIR, MODELS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          11,
    'figure.dpi':         120,
    'savefig.dpi':        200,
    'savefig.bbox':       'tight',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
})

MODEL_COLORS = {
    'BERT':                '#1565C0',
    'GRU':                 '#6A1B9A',
    'CNN_1D':              '#AD1457',
    'LSTM':                '#00695C',
    'Logistic_Regression': '#E65100',
    'Ridge_Classifier':    '#BF360C',
    'SVM_RBF':             '#4E342E',
    'SVM_Calibrated':      '#37474F',
    'LinearSVC':           '#263238',
    'Random_Forest':       '#2E7D32',
    'Extra_Trees':         '#558B2F',
    'XGBoost':             '#F9A825',
    'LightGBM':            '#00838F',
    'KNN':                 '#6D4C41',
    'Decision_Tree':       '#8D6E63',
    'SGD':                 '#546E7A',
    'Naive_Bayes':         '#7B1FA2',
}

print(f"Device      : {DEVICE}")
print(f"VRAM        : {VRAM_GB:.1f} GB")
print(f"FP16        : {USE_FP16}")
print(f"BERT config : batch={BERT_BATCH} | "
      f"maxlen={BERT_MAXLEN} | accum={GRAD_ACCUM}")

Device      : cuda
VRAM        : 8.0 GB
FP16        : True
BERT config : batch=4 | maxlen=128 | accum=8


In [4]:
# ============================================================
# LOAD DATA — handles your exact CSV structure
# columns: author, cleaned_body, Model, source, Type_genre
# ============================================================

print("Loading data...")
df_raw = pd.read_csv(DATA_PATH)

print(f"\nShape   : {df_raw.shape}")
print(f"Columns : {df_raw.columns.tolist()}")
print(f"\nDtypes:\n{df_raw.dtypes}")
print(f"\nFirst 2 rows:")
print(df_raw.head(2).to_string())

# ── Inspect each column ───────────────────────────────────────
print(f"\n--- Column Value Counts ---")
for col in df_raw.columns:
    if df_raw[col].dtype == object:
        vc = df_raw[col].value_counts()
        print(f"\n'{col}' ({df_raw[col].nunique()} unique):")
        print(vc.head(10).to_string())

# ── Check for nulls ───────────────────────────────────────────
print(f"\n--- Missing Values ---")
print(df_raw.isnull().sum())

Loading data...

Shape   : (5220, 5)
Columns : ['author', 'cleaned_body', 'Model', 'source', 'Type_genre']

Dtypes:
author          object
cleaned_body    object
Model           object
source          object
Type_genre      object
dtype: object

First 2 rows:
  author                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [5]:
# ============================================================
# DATA PREPARATION
# Your columns:
#   author      → 'ai' or 'human'   → LABEL
#   cleaned_body → the text          → TEXT
#   Model        → which AI model
#   source       → 'ai' or 'human'
#   Type_genre   → 'essays','fiction','news'
# ============================================================

df = df_raw.copy()

# ── 1. Clean column names ─────────────────────────────────────
df.columns = df.columns.str.strip().str.lower()
print("Columns after cleaning:", df.columns.tolist())

# ── 2. Identify columns ───────────────────────────────────────
TEXT_COL  = 'cleaned_body'
LABEL_COL = 'label'           # we will create this
GENRE_COL = 'type_genre'
MODEL_COL = 'model'
AUTH_COL  = 'author'

# ── 3. Create binary label ────────────────────────────────────
# author: 'ai' → 1,  'human' → 0
df[LABEL_COL] = (df[AUTH_COL].str.strip().str.lower() == 'ai'
                 ).astype(int)

print(f"\nLabel distribution:")
print(df[LABEL_COL].value_counts())
print(f"Balance ratio: "
      f"{df[LABEL_COL].value_counts().min() / len(df):.3f}")

# ── 4. Clean text ─────────────────────────────────────────────
df[TEXT_COL] = (df[TEXT_COL]
                .fillna('')
                .astype(str)
                .str.strip())

# Remove empty texts
mask_empty = df[TEXT_COL].str.len() < 50
print(f"\nRemoving {mask_empty.sum()} rows with text < 50 chars")
df = df[~mask_empty].reset_index(drop=True)

# ── 5. Genre distribution ─────────────────────────────────────
print(f"\nGenre distribution:")
print(df[GENRE_COL].value_counts())

print(f"\nLabel × Genre:")
print(pd.crosstab(df[LABEL_COL], df[GENRE_COL]))

print(f"\nFinal shape: {df.shape}")
print(f"Total samples: {len(df)}")
print(f"  Human: {(df[LABEL_COL]==0).sum()}")
print(f"  AI   : {(df[LABEL_COL]==1).sum()}")

# ── 6. Extract arrays ─────────────────────────────────────────
TEXTS  = df[TEXT_COL].tolist()
LABELS = df[LABEL_COL].values
GENRES = df[GENRE_COL].tolist()

print("\n✅ Data prepared")

Columns after cleaning: ['author', 'cleaned_body', 'model', 'source', 'type_genre']

Label distribution:
label
1    2610
0    2610
Name: count, dtype: int64
Balance ratio: 0.500

Removing 0 rows with text < 50 chars

Genre distribution:
type_genre
news       1740
fiction    1740
essays     1740
Name: count, dtype: int64

Label × Genre:
type_genre  essays  fiction  news
label                            
0              870      870   870
1              870      870   870

Final shape: (5220, 6)
Total samples: 5220
  Human: 2610
  AI   : 2610

✅ Data prepared


In [6]:



# ============================================================
# EXTRACT 40 METRICS
# ============================================================

import re

METRIC_COLS_EXPECTED = [
    'm01_average_word_length',
    'm02_type_token_ratio',
    'm03_yules_k',
    'm04_hapax_legomena',
    'm05_lexical_density',
    'm06_average_sentence_length',
    'm07_sentence_length_std',
    'm08_complex_sentence_ratio',
    'm09_average_clause_depth',
    'm10_punct_.',
    'm10_punct_,',
    'm10_punct_;',
    'm10_punct_!',
    'm10_punct_?',
    'm11_punctuation_consistency',
    'm12_dash_semicolon_ratio',
    'm13_burrows_delta',
    'm14_cosine_similarity',
    'm15_jensen_shannon_divergence',
    'm16_pos_bigram_0',
    'm16_pos_bigram_1',
    'm16_pos_bigram_2',
    'm17_pronoun_first_person',
    'm17_pronoun_second_person',
    'm17_pronoun_third_person',
    'm18_modal_frequency',
    'm19_character_ngram_diversity',
    'm20_word_bigram_entropy',
    'm22_rare_ngram_coverage',
    'm23_perplexity_approximation',
    'm24_shannon_entropy',
    'm25_cross_entropy_loss',
    'm26_flesch_kincaid_grade',
    'm27_flesch_reading_ease',
    'm28_gunning_fog_index',
    'm29_discourse_marker_frequency',
    'm30_connective_diversity',
    'm31_liwc_authenticity_proxy',
    'm32_coreference_pattern_frequency',
    'm33_semantic_relatedness',
    'm34_mattr',
    'm35_honores_r',
    'm36_zipfian_alpha',
    'm36_zipfian_r_squared',
    'm37_colocation_patterns',
    'm38_grammar_error_rate',
    'm39_sentence_construction_uniformity',
    'm40_token_predictability_variance',
]

already_present = [c for c in METRIC_COLS_EXPECTED if c in df.columns]
missing_metrics = [c for c in METRIC_COLS_EXPECTED if c not in df.columns]

print(f"Already extracted : {len(already_present)}")
print(f"Need to extract   : {len(missing_metrics)}")

if missing_metrics:
    print("\nExtracting metrics from text...")
    # print("(This takes ~2-5 min depending on dataset size)")

    sys.path.insert(0, '.')
    from authorship_extractor import AuthorshipMetricsExtractor

    # ----------------------------------------------------------------
    # BUILD REFERENCE BASELINE  (human texts → token list)
    # M13 (Burrows' Delta), M14 (Cosine Similarity), and
    # M15 (Jensen-Shannon Divergence) all need this reference.
    # ----------------------------------------------------------------
    BASELINE_LABEL      = 'human'       # label in 'source' column
    BASELINE_N_DOCS     = 300           # how many human docs to blend
    MAX_BASELINE_TOKENS = 3000          # cap for speed

    print(f"\nBuilding '{BASELINE_LABEL}' reference baseline ...")

    baseline_texts = (
        df[df['source'] == BASELINE_LABEL]['cleaned_body']
        .dropna()
        .sample(n=min(BASELINE_N_DOCS, (df['source'] == BASELINE_LABEL).sum()),
                random_state=42)
        .tolist()
    )

    # Tokenise baseline — same regex used inside AuthorshipMetricsExtractor
    _raw_baseline_tokens = [
        t.lower()
        for txt in baseline_texts
        for t in re.findall(r'\b\w+\b', str(txt))
        if t.isalpha()
    ]
    REFERENCE_TOKENS = _raw_baseline_tokens[:MAX_BASELINE_TOKENS]

    print(f"Reference baseline: {len(REFERENCE_TOKENS)} tokens "
          f"from {len(baseline_texts)} documents.\n")

    # ----------------------------------------------------------------
    # PER-DOCUMENT EXTRACTION
    # ----------------------------------------------------------------
    def safe_extract(text, reference_tokens):
        """
        Extract all 40 metrics for one document.

        Parameters
        ----------
        text : str
            The document body.
        reference_tokens : list[str]
            Pre-built baseline token list.  Passed to extract_all_metrics()
            so that M13, M14, and M15 are computed.
        """
        try:
            ext = AuthorshipMetricsExtractor(str(text))

            # ← KEY FIX: pass reference_tokens so M13/M14/M15 are computed
            m = ext.extract_all_metrics(text2_tokens=reference_tokens)

            flat = {}
            for k, v in m.items():
                # Skip the raw dict metric (stored as count only)
                if k == 'm21_top_word_bigrams_count':
                    continue
                if isinstance(v, dict):
                    for sk, sv in v.items():
                        try:
                            flat[f'{k}_{sk}'] = float(sv)
                        except (TypeError, ValueError):
                            flat[f'{k}_{sk}'] = np.nan
                else:
                    try:
                        flat[k] = float(v)
                    except (TypeError, ValueError):
                        flat[k] = np.nan
            return flat

        except Exception as e:
            # Return empty dict on failure; columns will become NaN
            print(f"  [warn] extraction failed: {e}")
            return {}

    # ----------------------------------------------------------------
    # RUN EXTRACTION
    # ----------------------------------------------------------------
    extracted_rows = []
    for text in tqdm(TEXTS, desc='Extracting metrics'):
        extracted_rows.append(safe_extract(text, REFERENCE_TOKENS))

    metrics_df = pd.DataFrame(extracted_rows)

    # Merge new columns back into df (only columns not already present)
    for col in metrics_df.columns:
        if col not in df.columns:
            df[col] = metrics_df[col].values

    enriched_path = OUTPUT_DIR / 'data_with_metrics.csv'
    df.to_csv(enriched_path, index=False)
    print(f"\nSaved enriched data: {enriched_path}")

else:
    print("✅ All metrics already in DataFrame")

# ── Final feature lists ───────────────────────────────────────
ALL_40 = [c for c in METRIC_COLS_EXPECTED if c in df.columns]

# ============================================================
# CUSTOM FEATURE SETS
#
# Case 3 — 6 core universal metrics
# Case 2 — 14 core + shared pattern metrics
# Case 1 — 25 full handcrafted feature space
# ============================================================

# ── Case 3: 6 core universal features ────────────────────────
CASE3_RAW = [
    'm01_average_word_length',
    'm15_jensen_shannon_divergence',
    'm27_flesch_reading_ease',
    'm36_zipfian_alpha',
    'm36_zipfian_r_squared',
    'm39_sentence_construction_uniformity',
]

# ── Case 2: 14 features (core + shared patterns) ─────────────
CASE2_RAW = CASE3_RAW + [
    'm02_type_token_ratio',
    'm04_hapax_legomena',
    'm07_sentence_length_std',
    'm26_flesch_kincaid_grade',
    'm28_gunning_fog_index',
    'm34_mattr',
    'm35_honores_r',
    'm40_token_predictability_variance',
]

# ── Case 1: 25 features (full handcrafted space) ─────────────
CASE1_RAW = CASE2_RAW + [
    'm03_yules_k',
    'm10_punct_.',
    'm10_punct_,',
    'm11_punctuation_consistency',
    'm14_cosine_similarity',
    'm16_pos_bigram_2',
    'm20_word_bigram_entropy',
    'm23_perplexity_approximation',
    'm24_shannon_entropy',
    'm25_cross_entropy_loss',
    'm34_mattr',   
]

# Filter to only columns that actually exist in df
CASE1 = [c for c in CASE1_RAW if c in df.columns]
CASE2 = [c for c in CASE2_RAW if c in df.columns]
CASE3 = [c for c in CASE3_RAW if c in df.columns]

# ── Verify counts and report any missing ──────────────────────
def report_case(name, requested, available):
    missing = [c for c in requested if c not in available]
    print(f"  {name}: requested={len(requested)} | "
          f"available={len(available)}", end='')
    if missing:
        print(f" | ⚠️  missing: {missing}")
    else:
        print(" ✅")

print("\nCustom feature set summary:")
report_case('Case1 (25  features)', CASE1_RAW, CASE1)
report_case('Case2 (14 features)', CASE2_RAW, CASE2)
report_case('Case3 (6 features)', CASE3_RAW, CASE3)
report_case('All_40             ', ALL_40,    ALL_40)

FEATURE_SETS = {
    'Case3_6':  CASE3,
    'Case2_14': CASE2,
    'Case1_25': CASE1,
    'All_40':   ALL_40,
}

print("\nFeature sets (final):")
for n, f in FEATURE_SETS.items():
    print(f"  {n:<12}: {len(f)} features")

print("\nSample feature values — Case1 (first row):")
print(df[CASE1].head(1).to_string())

# ── Sanity-check for M15 ────────────────────────
if 'm15_jensen_shannon_divergence' in df.columns:
    null_pct = df['m15_jensen_shannon_divergence'].isna().mean() * 100
    sample   = df['m15_jensen_shannon_divergence'].dropna().head(3).tolist()
    print(f"\nM15 null rate : {null_pct:.1f}%")
    print(f"M15 sample    : {[f'{v:.4f}' for v in sample]}")

Already extracted : 0
Need to extract   : 48

Extracting metrics from text...

Building 'human' reference baseline ...
Reference baseline: 3000 tokens from 300 documents.



Extracting metrics: 100%|██████████| 5220/5220 [05:29<00:00, 15.85it/s]



Saved enriched data: ml_results\data_with_metrics.csv

Custom feature set summary:
  Case1 (25  features): requested=25 | available=25 ✅
  Case2 (14 features): requested=14 | available=14 ✅
  Case3 (6 features): requested=6 | available=6 ✅
  All_40             : requested=48 | available=48 ✅

Feature sets (final):
  Case3_6     : 6 features
  Case2_14    : 14 features
  Case1_25    : 25 features
  All_40      : 48 features

Sample feature values — Case1 (first row):
   m01_average_word_length  m15_jensen_shannon_divergence  m27_flesch_reading_ease  m36_zipfian_alpha  m36_zipfian_r_squared  m39_sentence_construction_uniformity  m02_type_token_ratio  m04_hapax_legomena  m07_sentence_length_std  m26_flesch_kincaid_grade  m28_gunning_fog_index  m34_mattr  m35_honores_r  m40_token_predictability_variance  m03_yules_k  m10_punct_.  m10_punct_,  m11_punctuation_consistency  m14_cosine_similarity  m16_pos_bigram_2  m20_word_bigram_entropy  m23_perplexity_approximation  m24_shannon_entropy  m2

In [7]:
# ============================================================
# LOAD EXTERNAL TEST SET
# ============================================================

EXTERNAL_TEST_PATH = r'testing.csv'

print("Loading external test set...")
df_test_raw = pd.read_csv(EXTERNAL_TEST_PATH)

print(f"\nExternal test shape   : {df_test_raw.shape}")
print(f"External test columns : {df_test_raw.columns.tolist()}")

# ── Apply same cleaning pipeline as main df ───────────────────
df_test_ext = df_test_raw.copy()
df_test_ext.columns = df_test_ext.columns.str.strip().str.lower()

# Binary label
df_test_ext[LABEL_COL] = (
    df_test_ext[AUTH_COL].str.strip().str.lower() == 'ai'
).astype(int)

# Clean text
df_test_ext[TEXT_COL] = (
    df_test_ext[TEXT_COL]
    .fillna('')
    .astype(str)
    .str.strip()
)

# Remove short texts
mask_short_test = df_test_ext[TEXT_COL].str.len() < 50
print(f"Removing {mask_short_test.sum()} short rows from external test")
df_test_ext = df_test_ext[~mask_short_test].reset_index(drop=True)

print(f"\nExternal test label distribution:")
print(df_test_ext[LABEL_COL].value_counts())
print(f"\nExternal test genre distribution:")
print(df_test_ext[GENRE_COL].value_counts())
print(f"\nExternal test final shape: {df_test_ext.shape}")

# ── Extract metrics for external test set ─────────────────────
print("\nChecking metrics for external test set...")

missing_ext = [c for c in METRIC_COLS_EXPECTED
               if c not in df_test_ext.columns]

print(f"Need to extract : {len(missing_ext)} metrics")

if missing_ext:
    print("Extracting metrics for external test set...")
    TEXTS_EXT_RAW = df_test_ext[TEXT_COL].tolist()

    extracted_ext = []
    for text in tqdm(TEXTS_EXT_RAW, desc='Extracting metrics (ext test)'):
        extracted_ext.append(safe_extract(text, REFERENCE_TOKENS))

    metrics_ext_df = pd.DataFrame(extracted_ext)

    for col in metrics_ext_df.columns:
        if col not in df_test_ext.columns:
            df_test_ext[col] = metrics_ext_df[col].values

    enriched_test_path = OUTPUT_DIR / 'external_test_with_metrics.csv'
    df_test_ext.to_csv(enriched_test_path, index=False)
    print(f"Saved enriched external test: {enriched_test_path}")
else:
    print("✅ All metrics already in external test DataFrame")

# ── External test arrays ──────────────────────────────────────
TEXTS_EXT  = df_test_ext[TEXT_COL].tolist()
LABELS_EXT = df_test_ext[LABEL_COL].values
GENRES_EXT = df_test_ext[GENRE_COL].tolist()

# ============================================================
# STRATIFIED SPLIT  80 / 20  (train / val)  — main df only
# ============================================================

y   = LABELS
idx = np.arange(len(y))

# 80% train | 20% val  (from main df)
idx_tr, idx_va = train_test_split(
    idx,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# ── Point test variables to external set ──────────────────────
idx_te     = np.arange(len(LABELS_EXT))
y_test     = LABELS_EXT
texts_test = TEXTS_EXT

# Train & val from main df
texts_train = [TEXTS[i] for i in idx_tr]
texts_val   = [TEXTS[i] for i in idx_va]

y_train = y[idx_tr]
y_val   = y[idx_va]

print("\nSplit summary:")
print(f"  Train : {len(idx_tr):5d}  "
      f"(AI={y_train.sum():4d} | "
      f"Human={(y_train==0).sum():4d})")
print(f"  Val   : {len(idx_va):5d}  "
      f"(AI={y_val.sum():4d} | "
      f"Human={(y_val==0).sum():4d})")
print(f"  Test  : {len(idx_te):5d}  "
      f"(AI={y_test.sum():4d} | "
      f"Human={(y_test==0).sum():4d})  ← external file")

# ── Helper: build scaled feature arrays ───────────────────────
def get_xy(feature_cols):
    """
    Return scaled X splits and the fitted scaler.

    Train + Val  →  main df   (idx_tr, idx_va)
    Test         →  df_test_ext (external file)
    Scaler is fit on training rows only — no leakage.
    """
    valid = [c for c in feature_cols if c in df.columns]

    # ── Train & val from main df ───────────────────────────────
    X_main       = df[valid].copy()
    train_median = X_main.iloc[idx_tr].median()    # fit on train only
    X_main       = X_main.fillna(train_median).values.astype(np.float32)

    X_tr = X_main[idx_tr]
    X_va = X_main[idx_va]

    # ── Test from external df ──────────────────────────────────
    X_ext = df_test_ext.reindex(columns=valid).copy()
    X_ext = X_ext.fillna(train_median).values.astype(np.float32)
    X_te  = X_ext

    # ── Scale (fit on train only) ──────────────────────────────
    scaler  = RobustScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_va_sc = scaler.transform(X_va)
    X_te_sc = scaler.transform(X_te)

    return X_tr_sc, X_va_sc, X_te_sc, scaler, valid

print("\n✅ Split ready — external test file loaded & aligned")

Loading external test set...

External test shape   : (792, 5)
External test columns : ['author', 'cleaned_body', 'Model', 'source', 'Type_genre']
Removing 0 short rows from external test

External test label distribution:
label
1    396
0    396
Name: count, dtype: int64

External test genre distribution:
type_genre
news       264
essays     264
fiction    264
Name: count, dtype: int64

External test final shape: (792, 6)

Checking metrics for external test set...
Need to extract : 48 metrics
Extracting metrics for external test set...


Extracting metrics (ext test): 100%|██████████| 792/792 [00:49<00:00, 15.97it/s]


Saved enriched external test: ml_results\external_test_with_metrics.csv

Split summary:
  Train :  4176  (AI=2088 | Human=2088)
  Val   :  1044  (AI= 522 | Human= 522)
  Test  :   792  (AI= 396 | Human= 396)  ← external file

✅ Split ready — external test file loaded & aligned


In [8]:
# ============================================================
# METRICS HELPER
# ============================================================

def compute_metrics(y_true, y_pred, y_prob=None):
    result = {
        'accuracy':  round(float(
            accuracy_score(y_true, y_pred)), 4),
        'precision': round(float(
            precision_score(y_true, y_pred,
                            zero_division=0)), 4),
        'recall':    round(float(
            recall_score(y_true, y_pred,
                         zero_division=0)), 4),
        'f1':        round(float(
            f1_score(y_true, y_pred,
                     zero_division=0)), 4),
    }
    if y_prob is not None and len(y_prob) > 0:
        try:
            result['roc_auc'] = round(float(
                roc_auc_score(y_true, y_prob)), 4)
            result['avg_precision'] = round(float(
                average_precision_score(
                    y_true, y_prob)), 4)
        except Exception:
            result['roc_auc']       = np.nan
            result['avg_precision'] = np.nan
    else:
        result['roc_auc']       = np.nan
        result['avg_precision'] = np.nan
    return result


def get_prob(pipe, X):
    """Get class-1 probability safely."""
    try:
        return pipe.predict_proba(X)[:, 1]
    except AttributeError:
        try:
            sc = pipe.decision_function(X)
            lo, hi = sc.min(), sc.max()
            return (sc - lo) / (hi - lo + 1e-10)
        except Exception:
            return None


def empty_cv():
    return {k: np.nan for k in [
        'cv_acc_mean','cv_acc_std',
        'cv_auc_mean','cv_auc_std',
        'cv_f1_mean', 'cv_f1_std']}


print("✅ Metrics helper ready")

✅ Metrics helper ready


In [9]:
# ============================================================
# CLASSICAL ML MODELS + HP GRIDS
# ============================================================

def get_classical_models():
    """Return dict: name → (model, use_scaler, hp_grid)"""
    return {

        'Logistic_Regression': (
            LogisticRegression(
                random_state=SEED, max_iter=3000,
                class_weight='balanced'),
            True,
            {'clf__C':      [0.001,0.01,0.1,1,10,100],
             'clf__solver': ['lbfgs','liblinear']}),

        'Ridge_Classifier': (
            RidgeClassifier(),
            True,
            {'clf__alpha': [0.001,0.01,0.1,1,10,100]}),

        'SVM_RBF': (
            SVC(kernel='rbf', probability=True,
                class_weight='balanced',
                random_state=SEED),
            True,
            {'clf__C':     [0.1,1,10,100],
             'clf__gamma': ['scale','auto']}),

        'SVM_Calibrated': (
            CalibratedClassifierCV(
                LinearSVC(class_weight='balanced',
                          random_state=SEED,
                          max_iter=5000)),
            True,
            {'clf__estimator__C': [0.01,0.1,1,10]}),

        'LinearSVC': (
            LinearSVC(class_weight='balanced',
                      random_state=SEED,
                      max_iter=5000),
            True,
            {'clf__C': [0.001,0.01,0.1,1,10]}),

        'Random_Forest': (
            RandomForestClassifier(
                random_state=SEED, n_jobs=-1,
                class_weight='balanced'),
            True,
            {'clf__n_estimators':      [100,200,300],
             'clf__max_depth':         [None,10,20],
             'clf__min_samples_split': [2,5,10],
             'clf__max_features':      ['sqrt','log2']}),

        'Extra_Trees': (
            ExtraTreesClassifier(
                random_state=SEED, n_jobs=-1,
                class_weight='balanced'),
            True,
            {'clf__n_estimators': [100,200,300],
             'clf__max_depth':    [None,10,20],
             'clf__max_features': ['sqrt','log2']}),

        'XGBoost': (
            xgb.XGBClassifier(
                eval_metric='logloss',
                random_state=SEED, n_jobs=-1,
                verbosity=0),
            True,
            {'clf__n_estimators':     [100,200,300],
             'clf__max_depth':        [3,5,7],
             'clf__learning_rate':    [0.01,0.05,0.1],
             'clf__subsample':        [0.8,1.0],
             'clf__colsample_bytree': [0.8,1.0]}),

        'LightGBM': (
            lgb.LGBMClassifier(
                random_state=SEED, n_jobs=-1,
                verbose=-1),
            True,
            {'clf__n_estimators':  [100,200,300],
             'clf__max_depth':     [-1,10,20],
             'clf__learning_rate': [0.01,0.05,0.1],
             'clf__num_leaves':    [31,63,127]}),

        'KNN': (
            KNeighborsClassifier(n_jobs=-1),
            True,
            {'clf__n_neighbors': [3,5,7,11,15],
             'clf__weights':     ['uniform','distance'],
             'clf__metric':      ['minkowski','euclidean']}),

        'Decision_Tree': (
            DecisionTreeClassifier(random_state=SEED),
            True,
            {'clf__max_depth':         [None,5,10,20],
             'clf__min_samples_split': [2,5,10],
             'clf__criterion':         ['gini','entropy']}),

        'SGD': (
            SGDClassifier(
                random_state=SEED,
                class_weight='balanced'),
            True,
            {'clf__loss':    ['hinge','log_loss',
                              'modified_huber'],
             'clf__alpha':   [0.0001,0.001,0.01],
             'clf__max_iter':[1000]}),

        'Naive_Bayes': (
            MultinomialNB(),
            False,              # no scaler
            {'clf__alpha': [0.01,0.1,0.5,1.0,2.0]}),
    }


def train_classical(name, model, use_scaler, hp_grid,
                    X_tr, y_tr, X_te, y_te,
                    fs_name):
    """Train one classical model with HP tuning."""
    t0 = time.time()

    if name == 'Naive_Bayes':
        # NB needs non-negative features
        mm   = MinMaxScaler()
        X_tr = mm.fit_transform(np.abs(X_tr))
        X_te = mm.transform(np.abs(X_te))
        pipe = Pipeline([('clf', model)])
    else:
        steps = ([('scaler', RobustScaler())]
                 if use_scaler else [])
        steps.append(('clf', model))
        pipe = Pipeline(steps)

    # ── Randomised HP search ──────────────────────────────
    best_params = {}
    if hp_grid:
        inner_cv = StratifiedKFold(
            n_splits=3, shuffle=True, random_state=SEED)
        search = RandomizedSearchCV(
            pipe, hp_grid,
            n_iter=N_ITER_SEARCH,
            cv=inner_cv,
            scoring='roc_auc',
            n_jobs=-1,
            random_state=SEED,
            refit=True,
            error_score=0.0,
        )
        try:
            search.fit(X_tr, y_tr)
            pipe        = search.best_estimator_
            best_params = search.best_params_
        except Exception as e:
            print(f"      HP search failed ({e}), defaults")
            pipe.fit(X_tr, y_tr)
    else:
        pipe.fit(X_tr, y_tr)

    # ── Predictions ───────────────────────────────────────
    y_pred = pipe.predict(X_te)
    y_prob = get_prob(pipe, X_te)

    # ── Cross-validation ──────────────────────────────────
    X_full = np.vstack([X_tr, X_te])
    y_full = np.concatenate([y_tr, y_te])
    outer_cv = StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    try:
        cv_res = cross_validate(
            pipe, X_full, y_full,
            cv=outer_cv,
            scoring=['accuracy','roc_auc','f1'],
            n_jobs=-1)
        cv_info = {
            'cv_acc_mean': round(
                cv_res['test_accuracy'].mean(), 4),
            'cv_acc_std':  round(
                cv_res['test_accuracy'].std(),  4),
            'cv_auc_mean': round(
                cv_res['test_roc_auc'].mean(),  4),
            'cv_auc_std':  round(
                cv_res['test_roc_auc'].std(),   4),
            'cv_f1_mean':  round(
                cv_res['test_f1'].mean(),        4),
            'cv_f1_std':   round(
                cv_res['test_f1'].std(),         4),
        }
    except Exception:
        cv_info = empty_cv()

    metrics   = compute_metrics(y_test, y_pred, y_prob)
    elapsed   = time.time() - t0

    print(f"    Acc={metrics['accuracy']:.4f} | "
          f"AUC={metrics['roc_auc']:.4f} | "
          f"F1={metrics['f1']:.4f} | "
          f"CV-AUC={cv_info['cv_auc_mean']:.4f} | "
          f"t={elapsed:.1f}s")

    return {
        'model':        name,
        'feature_set':  fs_name,
        'model_type':   'Classical ML',
        'best_params':  str(best_params),
        'train_time_s': round(elapsed, 2),
        **cv_info,
        **metrics,
        'y_test': y_test.tolist(),
        'y_pred': y_pred.tolist(),
        'y_prob': (y_prob.tolist()
                   if y_prob is not None else []),
    }, pipe


print("✅ Classical ML engine ready")

✅ Classical ML engine ready


In [10]:
# ============================================================
# PYTORCH DL MODELS
# ============================================================

class FeatDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):        return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class GRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden=128,
                 layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128), nn.ReLU(),
            nn.Dropout(dropout))
        self.gru  = nn.GRU(
            128, hidden, layers, batch_first=True,
            dropout=dropout if layers>1 else 0,
            bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden*2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 2))

    def forward(self, x):
        x = self.proj(x).unsqueeze(1)
        _, h = self.gru(x)
        return self.head(
            torch.cat([h[-2], h[-1]], dim=1))


class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim, filters=128,
                 dropout=0.3):
        super().__init__()
        self.proj  = nn.Linear(input_dim, 128)
        self.conv1 = nn.Conv1d(1, filters, 3, padding=1)
        self.conv2 = nn.Conv1d(filters, filters,
                                5, padding=2)
        self.conv3 = nn.Conv1d(filters, filters,
                                7, padding=3)
        self.bn1   = nn.BatchNorm1d(filters)
        self.bn2   = nn.BatchNorm1d(filters)
        self.bn3   = nn.BatchNorm1d(filters)
        self.pool  = nn.AdaptiveMaxPool1d(1)
        self.head  = nn.Sequential(
            nn.Linear(filters*3, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, 2))

    def forward(self, x):
        x  = self.proj(x).unsqueeze(1)
        c1 = self.pool(
            F.relu(self.bn1(self.conv1(x)))).squeeze(-1)
        c2 = self.pool(
            F.relu(self.bn2(self.conv2(x)))).squeeze(-1)
        c3 = self.pool(
            F.relu(self.bn3(self.conv3(x)))).squeeze(-1)
        return self.head(
            torch.cat([c1, c2, c3], dim=1))


class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden=128,
                 layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128), nn.ReLU(),
            nn.Dropout(dropout))
        self.lstm = nn.LSTM(
            128, hidden, layers, batch_first=True,
            dropout=dropout if layers>1 else 0,
            bidirectional=True)
        self.attn = nn.Linear(hidden*2, 1)
        self.head = nn.Sequential(
            nn.Linear(hidden*2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 2))

    def forward(self, x):
        x   = self.proj(x).unsqueeze(1)
        out, _ = self.lstm(x)
        w   = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.head(ctx)


def train_dl(model, model_name, fs_name,
             X_tr, y_tr, X_va, y_va, X_te, y_te):
    """Generic train loop for GRU / CNN / LSTM."""
    t0    = time.time()
    pin   = torch.cuda.is_available()

    tr_ld = DataLoader(
        FeatDataset(X_tr, y_tr),
        batch_size=DL_BATCH, shuffle=True,
        pin_memory=pin)
    va_ld = DataLoader(
        FeatDataset(X_va, y_va),
        batch_size=DL_BATCH*2, pin_memory=pin)
    te_ld = DataLoader(
        FeatDataset(X_te, y_te),
        batch_size=DL_BATCH*2)

    model     = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), lr=DL_LR,
        weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS_DL)
    amp_sc    = GradScaler() if USE_FP16 else None

    best_val  = float('inf')
    best_wts  = None
    patience_ = 0
    history   = {
        'train_loss':[], 'val_loss':[], 'val_acc':[]}

    for epoch in range(MAX_EPOCHS_DL):
        # ── train ─────────────────────────────────────────
        model.train()
        tl = 0.0
        for Xb, yb in tr_ld:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            if USE_FP16:
                with autocast():
                    loss = criterion(model(Xb), yb)
                amp_sc.scale(loss).backward()
                amp_sc.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0)
                amp_sc.step(optimizer)
                amp_sc.update()
            else:
                loss = criterion(model(Xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0)
                optimizer.step()
            tl += loss.item()
        scheduler.step()

        # ── validate ──────────────────────────────────────
        model.eval()
        vl, preds, labs = 0.0, [], []
        with torch.no_grad():
            for Xb, yb in va_ld:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                if USE_FP16:
                    with autocast():
                        out = model(Xb)
                else:
                    out = model(Xb)
                vl   += criterion(out, yb).item()
                preds += out.argmax(1).cpu().tolist()
                labs  += yb.cpu().tolist()

        avg_tl = tl / len(tr_ld)
        avg_vl = vl / len(va_ld)
        vacc   = accuracy_score(labs, preds)
        history['train_loss'].append(avg_tl)
        history['val_loss'].append(avg_vl)
        history['val_acc'].append(vacc)

        if avg_vl < best_val:
            best_val  = avg_vl
            best_wts  = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()}
            patience_ = 0
        else:
            patience_ += 1
            if patience_ >= PATIENCE:
                break

    # ── test ──────────────────────────────────────────────
    model.load_state_dict(best_wts)
    model.to(DEVICE).eval()
    probs, preds, labs = [], [], []

    with torch.no_grad():
        for Xb, yb in te_ld:
            Xb = Xb.to(DEVICE)
            if USE_FP16:
                with autocast():
                    out = model(Xb)
            else:
                out = model(Xb)
            probs += torch.softmax(
                out,1)[:,1].cpu().tolist()
            preds += out.argmax(1).cpu().tolist()
            labs  += yb.tolist()

    y_true = np.array(labs)
    y_pred = np.array(preds)
    y_prob = np.array(probs)

    metrics = compute_metrics(y_true, y_pred, y_prob)
    elapsed = time.time() - t0

    print(f"    Acc={metrics['accuracy']:.4f} | "
          f"AUC={metrics['roc_auc']:.4f} | "
          f"F1={metrics['f1']:.4f} | "
          f"t={elapsed:.1f}s")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return {
        'model':        model_name,
        'feature_set':  fs_name,
        'model_type':   'Deep Learning',
        'best_params':  'hidden=128,layers=2,dropout=0.3',
        'train_time_s': round(elapsed, 2),
        **empty_cv(),
        **metrics,
        'y_test':   y_true.tolist(),
        'y_pred':   y_pred.tolist(),
        'y_prob':   y_prob.tolist(),
        'history':  history,
    }, model


print("✅ DL models ready")

✅ DL models ready


In [11]:
# ============================================================
# BERT — GPU OPTIMISED FOR 8 GB VRAM
# ============================================================

class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        enc = tokenizer(
            list(texts),
            max_length=BERT_MAXLEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt')
        self.ids   = enc['input_ids']
        self.masks = enc['attention_mask']
        self.labs  = torch.tensor(
            list(labels), dtype=torch.long)

    def __len__(self):        return len(self.labs)
    def __getitem__(self, i):
        return self.ids[i], self.masks[i], self.labs[i]


def train_bert(texts_tr, y_tr,
               texts_va, y_va,
               texts_te, y_te,
               fs_name='Text_Only'):

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    t0 = time.time()
    print(f"  BERT: batch={BERT_BATCH} | "
          f"maxlen={BERT_MAXLEN} | "
          f"accum={GRAD_ACCUM} | fp16={USE_FP16}")

    tokenizer = BertTokenizer.from_pretrained(
        'bert-base-uncased')
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased', num_labels=2,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1)
    model.gradient_checkpointing_enable()
    model = model.to(DEVICE)

    pin   = torch.cuda.is_available()

    print("  Tokenising train...")
    tr_ds = BERTDataset(texts_tr, y_tr, tokenizer)
    print("  Tokenising val...")
    va_ds = BERTDataset(texts_va, y_va, tokenizer)
    print("  Tokenising test...")
    te_ds = BERTDataset(texts_te, y_te, tokenizer)

    tr_ld = DataLoader(tr_ds, batch_size=BERT_BATCH,
                       shuffle=True, pin_memory=pin)
    va_ld = DataLoader(va_ds, batch_size=BERT_BATCH*2,
                       pin_memory=pin)
    te_ld = DataLoader(te_ds, batch_size=BERT_BATCH*2)

    # Optimiser with weight decay
    no_decay = ['bias','LayerNorm.weight']
    params = [
        {'params': [p for n,p in model.named_parameters()
                    if not any(nd in n
                               for nd in no_decay)],
         'weight_decay': 0.01},
        {'params': [p for n,p in model.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer   = optim.AdamW(params, lr=BERT_LR)
    total_steps = (len(tr_ld)//GRAD_ACCUM)*MAX_EPOCHS_BERT
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06*total_steps),
        num_training_steps=total_steps)
    criterion  = nn.CrossEntropyLoss()
    amp_sc     = GradScaler() if USE_FP16 else None

    best_val  = float('inf')
    best_wts  = None
    patience_ = 0
    history   = {
        'train_loss':[], 'val_loss':[], 'val_acc':[]}

    for epoch in range(MAX_EPOCHS_BERT):
        # ── train ─────────────────────────────────────────
        model.train()
        tl = 0.0
        optimizer.zero_grad()
        for step, (ids, masks, labs) in enumerate(tr_ld):
            ids   = ids.to(DEVICE)
            masks = masks.to(DEVICE)
            labs  = labs.to(DEVICE)

            if USE_FP16:
                with autocast():
                    out  = model(input_ids=ids,
                                  attention_mask=masks)
                    loss = criterion(
                        out.logits, labs) / GRAD_ACCUM
                amp_sc.scale(loss).backward()
                if (step+1) % GRAD_ACCUM == 0:
                    amp_sc.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(
                        model.parameters(), 1.0)
                    amp_sc.step(optimizer)
                    amp_sc.update()
                    scheduler.step()
                    optimizer.zero_grad()
            else:
                out  = model(input_ids=ids,
                              attention_mask=masks)
                loss = criterion(
                    out.logits, labs) / GRAD_ACCUM
                loss.backward()
                if (step+1) % GRAD_ACCUM == 0:
                    nn.utils.clip_grad_norm_(
                        model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
            tl += loss.item() * GRAD_ACCUM

        # ── validate ──────────────────────────────────────
        model.eval()
        vl, preds, labs_ = 0.0, [], []
        with torch.no_grad():
            for ids, masks, labs in va_ld:
                ids, masks = (ids.to(DEVICE),
                               masks.to(DEVICE))
                labs_d = labs.to(DEVICE)
                if USE_FP16:
                    with autocast():
                        out = model(input_ids=ids,
                                     attention_mask=masks)
                else:
                    out = model(input_ids=ids,
                                 attention_mask=masks)
                vl   += criterion(
                    out.logits, labs_d).item()
                preds += out.logits.argmax(
                    1).cpu().tolist()
                labs_ += labs.tolist()

        avg_tl = tl / len(tr_ld)
        avg_vl = vl / len(va_ld)
        vacc   = accuracy_score(labs_, preds)
        history['train_loss'].append(avg_tl)
        history['val_loss'].append(avg_vl)
        history['val_acc'].append(vacc)

        print(f"  Ep {epoch+1}/{MAX_EPOCHS_BERT} | "
              f"train={avg_tl:.4f} | "
              f"val={avg_vl:.4f} | "
              f"acc={vacc:.4f}")

        if avg_vl < best_val:
            best_val  = avg_vl
            best_wts  = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()}
            patience_ = 0
        else:
            patience_ += 1
            if patience_ >= PATIENCE:
                print(f"  Early stop ep {epoch+1}")
                break

    # ── test ──────────────────────────────────────────────
    model.load_state_dict(best_wts)
    model.to(DEVICE).eval()
    probs, preds, labs_ = [], [], []

    with torch.no_grad():
        for ids, masks, labs in te_ld:
            ids, masks = ids.to(DEVICE), masks.to(DEVICE)
            if USE_FP16:
                with autocast():
                    out = model(input_ids=ids,
                                 attention_mask=masks)
            else:
                out = model(input_ids=ids,
                             attention_mask=masks)
            probs += torch.softmax(
                out.logits,1)[:,1].cpu().tolist()
            preds += out.logits.argmax(
                1).cpu().tolist()
            labs_ += labs.tolist()

    y_true  = np.array(labs_)
    y_pred  = np.array(preds)
    y_prob  = np.array(probs)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    elapsed = time.time() - t0

    torch.cuda.empty_cache(); gc.collect()

    print(f"  BERT done: "
          f"Acc={metrics['accuracy']:.4f} | "
          f"AUC={metrics['roc_auc']:.4f} | "
          f"F1={metrics['f1']:.4f} | "
          f"t={elapsed:.1f}s")

    return {
        'model':        'BERT',
        'feature_set':  fs_name,
        'model_type':   'Deep Learning',
        'best_params':  (f"lr={BERT_LR},"
                         f"batch={BERT_BATCH},"
                         f"maxlen={BERT_MAXLEN},"
                         f"fp16={USE_FP16}"),
        'train_time_s': round(elapsed, 2),
        **empty_cv(),
        **metrics,
        'y_test':  y_true.tolist(),
        'y_pred':  y_pred.tolist(),
        'y_prob':  y_prob.tolist(),
        'history': history,
    }, model


print("✅ BERT ready")

✅ BERT ready


In [12]:
# ============================================================
# MASTER TRAINING LOOP
#
# Scenario 1 → BERT on raw text          (Text_Only)
# Scenario 2 → All 40 metrics            (All_40)
# Scenario 3 → Case 3:  6 metrics   (Case3_6)
# Scenario 4 → Case 2: 14 metrics        (Case2_14)
# Scenario 5 → Case 1: 25 metrics        (Case1_25)
# ============================================================

ALL_RESULTS = []

# ── SCENARIO 1: BERT on raw text ──────────────────────────────
print("\n" + "="*60)
print("SCENARIO 1 — BERT (Text Only)")
print("="*60)
res, mdl = train_bert(
    texts_train, y_train,
    texts_val,   y_val,
    texts_test,  y_test,
    fs_name='Text_Only')
res['scenario'] = 1
ALL_RESULTS.append(res)
torch.save(mdl.state_dict(),
           MODELS_DIR / 'BERT_Text_Only.pt')
del mdl; gc.collect(); torch.cuda.empty_cache()

# ── SCENARIOS 2-5: Metrics + Classical ML + DL ────────────────
# Map: feature-set name → scenario number
# All_40 keeps scenario 2; custom cases get 3/4/5
SCENARIO_MAP = {
    'All_40':   2,
    'Case3_6':  5,
    'Case2_14': 4,
    'Case1_25': 3,
}

for fs_name, s_num in SCENARIO_MAP.items():
    feat_cols = FEATURE_SETS[fs_name]

    print(f"\n{'='*60}")
    print(f"SCENARIO {s_num} — {fs_name} "
          f"({len(feat_cols)} features)")
    print(f"  Features: {feat_cols}")
    print(f"{'='*60}")

    X_tr, X_va, X_te, scaler, valid = \
        get_xy(feat_cols)
    input_dim = X_tr.shape[1]

    joblib.dump(scaler,
                MODELS_DIR / f'scaler_{fs_name}.pkl')

    # ── Classical ML ──────────────────────────────────────
    for m_name, (m_obj, use_sc, hp) in \
            get_classical_models().items():
        print(f"\n  [{fs_name}] {m_name}")
        try:
            res, pipe = train_classical(
                m_name, m_obj, use_sc, hp,
                X_tr, y_train,
                X_te, y_test,
                fs_name)
            res['scenario'] = s_num
            ALL_RESULTS.append(res)
            joblib.dump(pipe, MODELS_DIR /
                        f'{m_name}_{fs_name}.pkl')
        except Exception as e:
            print(f"    ❌ {m_name} failed: {e}")

    # ── GRU ───────────────────────────────────────────────
    print(f"\n  [{fs_name}] GRU")
    res, mdl = train_dl(
        GRUClassifier(input_dim),
        'GRU', fs_name,
        X_tr, y_train, X_va, y_val, X_te, y_test)
    res['scenario'] = s_num
    ALL_RESULTS.append(res)
    torch.save(mdl.state_dict(),
               MODELS_DIR / f'GRU_{fs_name}.pt')
    del mdl

    # ── LSTM ──────────────────────────────────────────────
    print(f"\n  [{fs_name}] LSTM")
    res, mdl = train_dl(
        LSTMClassifier(input_dim),
        'LSTM', fs_name,
        X_tr, y_train, X_va, y_val, X_te, y_test)
    res['scenario'] = s_num
    ALL_RESULTS.append(res)
    torch.save(mdl.state_dict(),
               MODELS_DIR / f'LSTM_{fs_name}.pt')
    del mdl

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n✅ All training done — "
      f"{len(ALL_RESULTS)} experiments")



SCENARIO 1 — BERT (Text Only)
  BERT: batch=4 | maxlen=128 | accum=8 | fp16=True
  Tokenising train...
  Tokenising val...
  Tokenising test...
  Ep 1/5 | train=0.3516 | val=0.1017 | acc=0.9655
  Ep 2/5 | train=0.0903 | val=0.0907 | acc=0.9732
  Ep 3/5 | train=0.0384 | val=0.2170 | acc=0.9464
  Ep 4/5 | train=0.0221 | val=0.2252 | acc=0.9511
  Ep 5/5 | train=0.0081 | val=0.2411 | acc=0.9502
  BERT done: Acc=0.9646 | AUC=0.9942 | F1=0.9650 | t=7772.1s

SCENARIO 2 — All_40 (48 features)
  Features: ['m01_average_word_length', 'm02_type_token_ratio', 'm03_yules_k', 'm04_hapax_legomena', 'm05_lexical_density', 'm06_average_sentence_length', 'm07_sentence_length_std', 'm08_complex_sentence_ratio', 'm09_average_clause_depth', 'm10_punct_.', 'm10_punct_,', 'm10_punct_;', 'm10_punct_!', 'm10_punct_?', 'm11_punctuation_consistency', 'm12_dash_semicolon_ratio', 'm13_burrows_delta', 'm14_cosine_similarity', 'm15_jensen_shannon_divergence', 'm16_pos_bigram_0', 'm16_pos_bigram_1', 'm16_pos_bigram_

# check from below

In [13]:
# ============================================================
# BUILD RESULTS DATAFRAME & SAVE ALL TABLES
# ============================================================

def build_results_df(all_results):
    rows = []
    for r in all_results:
        rows.append({
            'scenario':      r.get('scenario',      -1),
            'feature_set':   r.get('feature_set',   ''),
            'model':         r.get('model',          ''),
            'model_type':    r.get('model_type',     ''),
            'accuracy':      r.get('accuracy',       np.nan),
            'precision':     r.get('precision',      np.nan),
            'recall':        r.get('recall',         np.nan),
            'f1':            r.get('f1',             np.nan),
            'roc_auc':       r.get('roc_auc',        np.nan),
            'avg_precision': r.get('avg_precision',  np.nan),
            'cv_acc_mean':   r.get('cv_acc_mean',    np.nan),
            'cv_acc_std':    r.get('cv_acc_std',     np.nan),
            'cv_auc_mean':   r.get('cv_auc_mean',    np.nan),
            'cv_auc_std':    r.get('cv_auc_std',     np.nan),
            'cv_f1_mean':    r.get('cv_f1_mean',     np.nan),
            'cv_f1_std':     r.get('cv_f1_std',      np.nan),
            'train_time_s':  r.get('train_time_s',   np.nan),
            'best_params':   r.get('best_params',    ''),
        })
    rdf = (pd.DataFrame(rows)
           .sort_values(['scenario','roc_auc'],
                        ascending=[True, False])
           .reset_index(drop=True))
    rdf['overall_rank'] = (rdf['roc_auc']
                           .rank(ascending=False)
                           .astype(int))
    return rdf


results_df = build_results_df(ALL_RESULTS)

# ── Human-readable scenario labels ────────────────────────────
SCENARIO_LABELS = {
    1: 'S1: BERT (Text Only)',
    2: 'S2: n=49 (ALL_40)',
    5: 'S5: n=6 (core)',
    4: 'S4: n=14 (mixed)',
    3: 'S3: n=25 (full)',
}
results_df['scenario_label'] = results_df[
    'scenario'].map(SCENARIO_LABELS)

# ── Save tables ───────────────────────────────────────────────
results_df.to_csv(
    TABLES_DIR / '01_master_results.csv', index=False)

# Best per scenario
(results_df
 .groupby('scenario', group_keys=False)
 .apply(lambda g: g.nlargest(1, 'roc_auc'))
 .reset_index(drop=True)
 .to_csv(TABLES_DIR / '02_best_per_scenario.csv',
         index=False))

# AUC pivot (model × feature_set)
(results_df
 .pivot_table(index='model', columns='feature_set',
              values='roc_auc', aggfunc='mean')
 .round(4)
 .to_csv(TABLES_DIR / '03_auc_pivot.csv'))

# F1 pivot
(results_df
 .pivot_table(index='model', columns='feature_set',
              values='f1', aggfunc='mean')
 .round(4)
 .to_csv(TABLES_DIR / '04_f1_pivot.csv'))

# DL vs ML summary
DL_MODELS = ['BERT','GRU','CNN_1D','LSTM']
results_df['model_family'] = results_df['model'].apply(
    lambda m: 'Deep Learning'
    if m in DL_MODELS else 'Classical ML')
(results_df
 .groupby(['model_family','feature_set'])
 [['accuracy','precision','recall','f1',
   'roc_auc','train_time_s']]
 .agg(['mean','std'])
 .round(4)
 .to_csv(TABLES_DIR / '05_dl_vs_ml.csv'))

# Per-scenario CSVs
for s in sorted(results_df['scenario'].unique()):
    sub = results_df[results_df['scenario']==s]
    sub.to_csv(
        TABLES_DIR / f'scenario_{s}_results.csv',
        index=False)

# Classification reports for top-5
top5_rows = results_df.nlargest(5, 'roc_auc')
cr_rows   = []
for _, row in top5_rows.iterrows():
    res = next(
        (r for r in ALL_RESULTS
         if r['model']==row['model']
         and r['feature_set']==row['feature_set']),
        None)
    if res is None: continue
    cr = classification_report(
        res['y_test'], res['y_pred'],
        target_names=['Human','AI'],
        output_dict=True)
    for cls, vals in cr.items():
        if isinstance(vals, dict):
            cr_rows.append({
                'model':       row['model'],
                'feature_set': row['feature_set'],
                'class':       cls,
                **{k: round(v,4)
                   for k,v in vals.items()}})
(pd.DataFrame(cr_rows)
 .to_csv(TABLES_DIR / '06_classification_reports.csv',
         index=False))

# ── Extra: feature-set membership reference table ────────────
feat_ref_rows = []
all_named = {
    'Case1_6':  CASE1,
    'Case2_14': CASE2,
    'Case3_25': CASE3,
    'All_40':   ALL_40,
}
for feat in ALL_40:
    feat_ref_rows.append({
        'feature':    feat,
        'in_Case1_6':  feat in CASE1,
        'in_Case2_14': feat in CASE2,
        'in_Case3_25': feat in CASE3,
        'in_All_40':   True,
    })
(pd.DataFrame(feat_ref_rows)
 .to_csv(TABLES_DIR / '07_feature_set_membership.csv',
         index=False))

print("✅ All tables saved")
print(f"\nTop-10 results:")
print(results_df[['overall_rank','model','feature_set',
                   'accuracy','precision','recall',
                   'f1','roc_auc']]
      .head(10).to_string(index=False))

✅ All tables saved

Top-10 results:
 overall_rank               model feature_set  accuracy  precision  recall     f1  roc_auc
            1                BERT   Text_Only    0.9646     0.9554  0.9747 0.9650   0.9942
            2            LightGBM      All_40    0.9381     0.9370  0.9394 0.9382   0.9856
            3             XGBoost      All_40    0.9356     0.9218  0.9520 0.9366   0.9852
            4                LSTM      All_40    0.9419     0.9289  0.9571 0.9428   0.9846
            5                 GRU      All_40    0.9381     0.9392  0.9369 0.9381   0.9828
            7             SVM_RBF      All_40    0.9394     0.9372  0.9419 0.9395   0.9823
           11 Logistic_Regression      All_40    0.9306     0.9128  0.9520 0.9320   0.9799
           13      SVM_Calibrated      All_40    0.9331     0.9153  0.9545 0.9345   0.9797
           14           LinearSVC      All_40    0.9331     0.9173  0.9520 0.9343   0.9795
           15                 SGD      All_40    0.929

In [18]:
# ============================================================
# VISUALISATIONS
# ============================================================

def save_fig(name):
    path = PLOTS_DIR / name
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {path.name}")


# ── 1. Heatmaps ───────────────────────────────────────────────
# Column order: most features → fewest
FS_ORDER = ['All_40','Case3_25','Case2_14','Case1_6',
            'Text_Only']

for metric in ['roc_auc', 'f1', 'accuracy']:
    pivot = (results_df
             .pivot_table(index='model',
                          columns='feature_set',
                          values=metric,
                          aggfunc='mean'))
    # Reorder columns to present order
    col_order = [c for c in FS_ORDER
                 if c in pivot.columns]
    pivot = pivot[col_order]
    pivot = pivot.sort_values(col_order[0],
                              ascending=False)

    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(pivot, annot=True, fmt='.3f',
                cmap='RdYlGn', vmin=0.5, vmax=1.0,
                linewidths=0.5, linecolor='white',
                ax=ax,
                cbar_kws={'label':  metric.upper(),
                           'shrink': 0.7})
    ax.set_title(
        f'Model × Feature Set — {metric.upper()}\n'
        f'(S3:n=25  S4:n=14  '
        f'S5:n=6  S2:n=49(ALL_40)  S1=TextOnly)',
        fontweight='bold', fontsize=13)
    ax.set_xlabel('Feature Set')
    ax.set_ylabel('Model')
    save_fig(f'heatmap_{metric}.png')


# ── 2. ROC curves ─────────────────────────────────────────────
groups_plot = {
    'Deep Learning':
        ['BERT','GRU','CNN_1D','LSTM'],
    'Linear & SVM':
        ['Logistic_Regression','Ridge_Classifier',
         'LinearSVC','SGD',
         'SVM_RBF','SVM_Calibrated'],
    'Ensemble & Boosting':
        ['Random_Forest','Extra_Trees',
         'XGBoost','LightGBM'],
    'Other Classifiers':
        ['KNN','Decision_Tree','Naive_Bayes'],
}

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for ax, (gname, mlist) in zip(
        axes.flatten(), groups_plot.items()):
    for mname in mlist:
        best = max(
            (r for r in ALL_RESULTS
             if r['model']==mname
             and len(r.get('y_prob',[]))>0),
            key=lambda r: r.get('roc_auc', 0),
            default=None)
        if best is None: continue
        fpr, tpr, _ = roc_curve(
            best['y_test'], best['y_prob'])
        auc_v = auc(fpr, tpr)
        col   = MODEL_COLORS.get(mname,'#9E9E9E')
        fs    = best['feature_set']
        ax.plot(fpr, tpr, lw=2, color=col,
                alpha=0.85,
                label=(f"{mname[:16]} "
                       f"[{fs[:8]}] "
                       f"AUC={auc_v:.3f}"))

    ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC — {gname}', fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower right')
    ax.grid(alpha=0.2)

plt.suptitle('ROC Curves (best feature set per model)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('roc_curves.png')


# ── 3. PR curves ──────────────────────────────────────────────
baseline_pr = y_test.mean()
fig, axes   = plt.subplots(2, 2, figsize=(15, 12))
for ax, (gname, mlist) in zip(
        axes.flatten(), groups_plot.items()):
    for mname in mlist:
        best = max(
            (r for r in ALL_RESULTS
             if r['model']==mname
             and len(r.get('y_prob',[]))>0),
            key=lambda r: r.get('avg_precision', 0),
            default=None)
        if best is None: continue
        prec, rec, _ = precision_recall_curve(
            best['y_test'], best['y_prob'])
        ap  = average_precision_score(
            best['y_test'], best['y_prob'])
        col = MODEL_COLORS.get(mname,'#9E9E9E')
        ax.plot(rec, prec, lw=2,
                color=col, alpha=0.85,
                label=f"{mname[:16]} AP={ap:.3f}")

    ax.axhline(baseline_pr, ls='--', color='gray',
               lw=1.5, alpha=0.5,
               label=f'Baseline={baseline_pr:.2f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve — {gname}',
                 fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower left')
    ax.grid(alpha=0.2)

plt.suptitle('Precision-Recall Curves',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('pr_curves.png')


# ── 4. Confusion matrices (top 6) ─────────────────────────────
top6 = results_df.nlargest(6, 'roc_auc')
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, (_, row) in enumerate(top6.iterrows()):
    ax  = axes.flatten()[i]
    res = next(
        (r for r in ALL_RESULTS
         if r['model']==row['model']
         and r['feature_set']==row['feature_set']),
        None)
    if res is None:
        ax.axis('off'); continue

    cm   = confusion_matrix(res['y_test'], res['y_pred'])
    disp = ConfusionMatrixDisplay(
        cm, display_labels=['Human','AI'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(
        f"{row['model'].replace('_',' ')}\n"
        f"[{row['feature_set']}] | "
        f"AUC={row['roc_auc']:.3f} | "
        f"F1={row['f1']:.3f}",
        fontsize=9, fontweight='bold')

plt.suptitle('Confusion Matrices — Top 6 Models',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('confusion_matrices.png')


# ── 5. Scenario comparison bar chart ─────────────────────────
met_list = ['accuracy','precision','recall',
            'f1','roc_auc']

# Palette: one colour per feature set
pal_sc = {
    'Text_Only': '#1565C0',
    'All_40':    '#2E7D32',
    'Case3_6':   '#6A1B9A',
    'Case2_14':  '#AD1457',
    'Case1_25':  '#E65100',
}

fig, axes = plt.subplots(1, 5, figsize=(24, 7))
for ax, met in zip(axes, met_list):
    grp = (results_df
           .groupby('feature_set')[met]
           .agg(['mean','std'])
           .reset_index()
           .sort_values('mean', ascending=True))
    cols = [pal_sc.get(fs,'#9E9E9E')
            for fs in grp['feature_set']]
    ax.barh(grp['feature_set'], grp['mean'],
            xerr=grp['std'], capsize=4,
            color=cols, alpha=0.85,
            edgecolor='white',
            error_kw={'elinewidth':1.5,
                      'ecolor':'black'})
    ax.set_xlabel(met.upper())
    ax.set_title(met.upper(), fontweight='bold')
    ax.set_xlim(0.4, 1.05)
    ax.grid(axis='x', alpha=0.25)
    for _, r in grp.iterrows():
        idx_pos = grp.index.get_loc(
            grp[grp['feature_set']==
                r['feature_set']].index[0])
        ax.text(r['mean']+0.005, idx_pos,
                f"{r['mean']:.3f}",
                va='center', fontsize=8)

plt.suptitle(
    'Performance by Feature Set (mean ± std, all models)\n'
    'n=6 core | n=14 mixed | '
    'n=25 full | All40=all metrics | Text=BERT only',
    fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('scenario_comparison.png')


# ── 6. Model ranking (overall, across all scenarios) ─────────
grp = (results_df
       .groupby('model')['roc_auc']
       .agg(['mean','std'])
       .reset_index()
       .sort_values('mean', ascending=True))

fig, ax = plt.subplots(figsize=(10, 10))
cols = [MODEL_COLORS.get(m,'#9E9E9E')
        for m in grp['model']]
ax.barh(grp['model'], grp['mean'],
        xerr=grp['std'], capsize=3,
        color=cols, alpha=0.85,
        edgecolor='white',
        error_kw={'elinewidth':1.5,'ecolor':'black'})
ax.axvline(0.9, color='green',  lw=2, ls='--',
           alpha=0.6, label='AUC=0.90')
ax.axvline(0.8, color='orange', lw=2, ls='--',
           alpha=0.6, label='AUC=0.80')
ax.set_xlabel('Mean ROC-AUC (all feature sets)',
              fontsize=12)
ax.set_title('Overall Model Ranking',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.25)
ax.set_xlim(0.4, 1.05)
plt.tight_layout()
save_fig('model_ranking.png')


# ── 7. DL learning curves ─────────────────────────────────────
dl_res = [r for r in ALL_RESULTS
          if 'history' in r
          and len(r['history']['train_loss'])>0]

if dl_res:
    ncols = 4
    nrows = int(np.ceil(len(dl_res)/ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(18, nrows*4))
    flat = (axes.flatten()
            if hasattr(axes,'flatten') else [axes])

    for i, r in enumerate(dl_res):
        ax  = flat[i]
        h   = r['history']
        ep  = range(1, len(h['train_loss'])+1)
        col = MODEL_COLORS.get(r['model'],'#333333')
        ax.plot(ep, h['train_loss'],
                color=col, lw=2, label='Train Loss')
        ax.plot(ep, h['val_loss'],
                color=col, lw=2, ls='--',
                label='Val Loss')
        ax2 = ax.twinx()
        ax2.plot(ep, h['val_acc'],
                 color='black', lw=1.5, ls=':',
                 label='Val Acc')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss', color=col)
        ax2.set_ylabel('Val Acc', color='black')
        ax2.set_ylim(0, 1)
        ax.set_title(
            f"{r['model']} [{r['feature_set']}]",
            fontweight='bold', fontsize=9)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7, loc='upper right')
        ax2.legend(fontsize=7, loc='lower right')

    for j in range(i+1, len(flat)):
        flat[j].axis('off')

    plt.suptitle('DL Learning Curves',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig('learning_curves.png')


# ── 8. Speed vs AUC ──────────────────────────────────────────
fs_markers = {
    'Text_Only': 'o',
    'All_40':    's',
    'Case3_6':   '^',
    'Case2_14':  'D',
    'Case1_25':  'v',
}

fig, ax = plt.subplots(figsize=(12, 8))
for _, row in results_df.iterrows():
    c = MODEL_COLORS.get(row['model'],'#9E9E9E')
    m = fs_markers.get(row['feature_set'],'o')
    ax.scatter(row['train_time_s'], row['roc_auc'],
               c=c, marker=m, s=90, alpha=0.75,
               edgecolors='white', linewidths=0.8)

ax.axhline(0.9, color='green',  lw=1.5, ls='--',
           alpha=0.5, label='AUC=0.90')
ax.axhline(0.8, color='orange', lw=1.5, ls='--',
           alpha=0.5, label='AUC=0.80')
ax.set_xlabel('Training Time (s)', fontsize=12)
ax.set_ylabel('ROC-AUC',           fontsize=12)
ax.set_title(
    'Speed vs Accuracy\n'
    'Colour=Model | Shape=Feature Set\n'
    '(▲=Case1-6  ◆=Case2-13  ▼=Case3-26'
    '  ■=All40  ●=TextOnly)',
    fontweight='bold')

handles = [mpatches.Patch(
    color=c, label=m.replace('_',' '))
    for m, c in MODEL_COLORS.items()
    if m in results_df['model'].unique()]
leg1 = ax.legend(handles=handles,
                  fontsize=7.5, loc='lower right',
                  title='Model', ncol=2)
ax.add_artist(leg1)
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.2)
plt.tight_layout()
save_fig('speed_vs_auc.png')


# ── 9. Radar chart ────────────────────────────────────────────
from math import pi
mets_r = ['accuracy','precision',
          'recall','f1','roc_auc']
grp_r  = (results_df
          .groupby('feature_set')[mets_r]
          .mean())
N      = len(mets_r)
angles = [pi*2*i/N for i in range(N)] + [0]

pal_r = {
    'Text_Only': '#1565C0',
    'All_40':    '#2E7D32',
    'Case3_6':   '#6A1B9A',
    'Case2_14':  '#AD1457',
    'Case1_25':  '#E65100',
}

fig, ax = plt.subplots(
    figsize=(9, 9), subplot_kw=dict(polar=True))
for fs, row in grp_r.iterrows():
    vals = row[mets_r].tolist() + [row[mets_r[0]]]
    col  = pal_r.get(fs,'#9E9E9E')
    ax.plot(angles, vals, lw=2.5,
            color=col, label=fs)
    ax.fill(angles, vals, color=col, alpha=0.10)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(
    [m.upper() for m in mets_r], fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(
    'Radar: Feature Set Comparison\n'
    '(Case1=6  Case2=13  Case3=26  All40  TextOnly)',
    fontsize=12, fontweight='bold', pad=22)
ax.legend(loc='upper right',
           bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
save_fig('radar_chart.png')


# ── 10. NEW: Per-scenario best-model bar chart ────────────────
best_per_s = (results_df
              .groupby(['scenario','scenario_label'],
                       group_keys=False)
              .apply(lambda g: g.nlargest(1, 'roc_auc'))
              .reset_index(drop=True)
              .sort_values('scenario'))

fig, ax = plt.subplots(figsize=(11, 6))
colors  = [MODEL_COLORS.get(m,'#9E9E9E')
           for m in best_per_s['model']]
bars = ax.bar(best_per_s['scenario_label'],
              best_per_s['roc_auc'],
              color=colors, alpha=0.85,
              edgecolor='white', width=0.6)
for bar, (_, row) in zip(bars, best_per_s.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{row['model'].replace('_',' ')}\n"
            f"AUC={row['roc_auc']:.3f}",
            ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Best ROC-AUC', fontsize=12)
ax.set_title('Best Model per Scenario',
             fontweight='bold', fontsize=13)
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
save_fig('best_per_scenario.png')


print("\n✅ All plots saved")


  Saved → heatmap_roc_auc.png
  Saved → heatmap_f1.png
  Saved → heatmap_accuracy.png
  Saved → roc_curves.png
  Saved → pr_curves.png
  Saved → confusion_matrices.png
  Saved → scenario_comparison.png
  Saved → model_ranking.png
  Saved → learning_curves.png
  Saved → speed_vs_auc.png
  Saved → radar_chart.png
  Saved → best_per_scenario.png

✅ All plots saved


In [19]:
# ============================================================
# BUILD RESULTS DATAFRAME & SAVE ALL TABLES
# ============================================================

def build_results_df(all_results):
    rows = []
    for r in all_results:
        rows.append({
            'scenario':      r.get('scenario',      -1),
            'feature_set':   r.get('feature_set',   ''),
            'model':         r.get('model',          ''),
            'model_type':    r.get('model_type',     ''),
            'accuracy':      r.get('accuracy',       np.nan),
            'precision':     r.get('precision',      np.nan),
            'recall':        r.get('recall',         np.nan),
            'f1':            r.get('f1',             np.nan),
            'roc_auc':       r.get('roc_auc',        np.nan),
            'avg_precision': r.get('avg_precision',  np.nan),
            'cv_acc_mean':   r.get('cv_acc_mean',    np.nan),
            'cv_acc_std':    r.get('cv_acc_std',     np.nan),
            'cv_auc_mean':   r.get('cv_auc_mean',    np.nan),
            'cv_auc_std':    r.get('cv_auc_std',     np.nan),
            'cv_f1_mean':    r.get('cv_f1_mean',     np.nan),
            'cv_f1_std':     r.get('cv_f1_std',      np.nan),
            'train_time_s':  r.get('train_time_s',   np.nan),
            'best_params':   r.get('best_params',    ''),
        })
    rdf = (pd.DataFrame(rows)
           .sort_values(['scenario','roc_auc'],
                        ascending=[True, False])
           .reset_index(drop=True))
    rdf['overall_rank'] = (rdf['roc_auc']
                           .rank(ascending=False)
                           .astype(int))
    return rdf


results_df = build_results_df(ALL_RESULTS)

# ── Save tables ───────────────────────────────────────────────
results_df.to_csv(
    TABLES_DIR / '01_master_results.csv', index=False)

# Best per scenario
(results_df
 .groupby('scenario', group_keys=False)
 .apply(lambda g: g.nlargest(1, 'roc_auc'))
 .reset_index(drop=True)
 .to_csv(TABLES_DIR / '02_best_per_scenario.csv',
         index=False))

# AUC pivot
(results_df
 .pivot_table(index='model', columns='feature_set',
              values='roc_auc', aggfunc='mean')
 .round(4)
 .to_csv(TABLES_DIR / '03_auc_pivot.csv'))

# F1 pivot
(results_df
 .pivot_table(index='model', columns='feature_set',
              values='f1', aggfunc='mean')
 .round(4)
 .to_csv(TABLES_DIR / '04_f1_pivot.csv'))

# DL vs ML summary
DL_MODELS = ['BERT','GRU','LSTM']
results_df['model_family'] = results_df['model'].apply(
    lambda m: 'Deep Learning'
    if m in DL_MODELS else 'Classical ML')
(results_df
 .groupby(['model_family','feature_set'])
 [['accuracy','precision','recall','f1',
   'roc_auc','train_time_s']]
 .agg(['mean','std'])
 .round(4)
 .to_csv(TABLES_DIR / '05_dl_vs_ml.csv'))

# Per-scenario CSVs
for s in sorted(results_df['scenario'].unique()):
    sub = results_df[results_df['scenario']==s]
    sub.to_csv(
        TABLES_DIR / f'scenario_{s}_results.csv',
        index=False)

# Classification reports for top-5
top5_rows = results_df.nlargest(5, 'roc_auc')
cr_rows   = []
for _, row in top5_rows.iterrows():
    res = next(
        (r for r in ALL_RESULTS
         if r['model']==row['model']
         and r['feature_set']==row['feature_set']),
        None)
    if res is None: continue
    cr = classification_report(
        res['y_test'], res['y_pred'],
        target_names=['Human','AI'],
        output_dict=True)
    for cls, vals in cr.items():
        if isinstance(vals, dict):
            cr_rows.append({
                'model':       row['model'],
                'feature_set': row['feature_set'],
                'class':       cls,
                **{k: round(v,4)
                   for k,v in vals.items()}})
(pd.DataFrame(cr_rows)
 .to_csv(TABLES_DIR / '06_classification_reports.csv',
         index=False))

print("✅ All tables saved")
print(f"\nTop-10 results:")
print(results_df[['overall_rank','model','feature_set',
                   'accuracy','precision','recall',
                   'f1','roc_auc']]
      .head(10).to_string(index=False))

✅ All tables saved

Top-10 results:
 overall_rank               model feature_set  accuracy  precision  recall     f1  roc_auc
            1                BERT   Text_Only    0.9646     0.9554  0.9747 0.9650   0.9942
            2            LightGBM      All_40    0.9381     0.9370  0.9394 0.9382   0.9856
            3             XGBoost      All_40    0.9356     0.9218  0.9520 0.9366   0.9852
            4                LSTM      All_40    0.9419     0.9289  0.9571 0.9428   0.9846
            5                 GRU      All_40    0.9381     0.9392  0.9369 0.9381   0.9828
            7             SVM_RBF      All_40    0.9394     0.9372  0.9419 0.9395   0.9823
           11 Logistic_Regression      All_40    0.9306     0.9128  0.9520 0.9320   0.9799
           13      SVM_Calibrated      All_40    0.9331     0.9153  0.9545 0.9345   0.9797
           14           LinearSVC      All_40    0.9331     0.9173  0.9520 0.9343   0.9795
           15                 SGD      All_40    0.929

In [21]:
# ============================================================
# FINAL SUMMARY REPORT
# ============================================================

best = results_df.nlargest(1, 'roc_auc').iloc[0]
top5 = results_df.nlargest(5, 'roc_auc')[
    ['overall_rank','model','feature_set',
     'accuracy','precision','recall',
     'f1','roc_auc','train_time_s']]

print("="*70)
print("  EXPERIMENT COMPLETE")
print("="*70)
print(f"  Experiments run  : {len(results_df)}")
print(f"  Best model       : {best['model']}")
print(f"  Best feature set : {best['feature_set']}")
print(f"  Accuracy         : {best['accuracy']:.4f}")
print(f"  Precision        : {best['precision']:.4f}")
print(f"  Recall           : {best['recall']:.4f}")
print(f"  F1 Score         : {best['f1']:.4f}")
print(f"  ROC-AUC          : {best['roc_auc']:.4f}")

print(f"\nBest model per scenario:")
for _, row in best_per_s.iterrows():
    print(f"  {row['scenario_label']:<28} "
          f"→ {row['model']:<22} "
          f"AUC={row['roc_auc']:.4f}")

print(f"\nTop-5 Models (overall):")
print(top5.to_string(index=False))

print(f"\nFeature set summary:")
print(f"  Case1_6  : {len(CASE1):2d} features — "
      f"{CASE1}")
print(f"  Case2_14 : {len(CASE2):2d} features — "
      f"{CASE2}")
print(f"  Case3_25 : {len(CASE3):2d} features — "
      f"{CASE3}")
print(f"  All_40   : {len(ALL_40):2d} features")

print(f"\nOutput files:")
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        kb = f.stat().st_size / 1024
        print(f"  {str(f.relative_to(OUTPUT_DIR)):<52}"
              f"  {kb:>8.1f} KB")
print("="*70)

  EXPERIMENT COMPLETE
  Experiments run  : 61
  Best model       : BERT
  Best feature set : Text_Only
  Accuracy         : 0.9646
  Precision        : 0.9554
  Recall           : 0.9747
  F1 Score         : 0.9650
  ROC-AUC          : 0.9942

Best model per scenario:
  S1: BERT (Text Only)         → BERT                   AUC=0.9942
  S2: n=49 (ALL_40)            → LightGBM               AUC=0.9856
  S3: n=25 (full)              → LSTM                   AUC=0.9826
  S4: n=14 (mixed)             → SVM_RBF                AUC=0.9635
  S5: n=6 (core)               → XGBoost                AUC=0.9278

Top-5 Models (overall):
 overall_rank    model feature_set  accuracy  precision  recall     f1  roc_auc  train_time_s
            1     BERT   Text_Only    0.9646     0.9554  0.9747 0.9650   0.9942       7772.14
            2 LightGBM      All_40    0.9381     0.9370  0.9394 0.9382   0.9856         33.20
            3  XGBoost      All_40    0.9356     0.9218  0.9520 0.9366   0.9852          

In [27]:
# # ============================================================
# # TOP-1 MODELS ACROSS ALL 5 SCENARIOS
# # Combined: ROC curve + PR curve + Confusion Matrix
# # ============================================================

# from pathlib import Path
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# from sklearn.metrics import (
#     roc_curve, auc,
#     precision_recall_curve, average_precision_score,
#     confusion_matrix, ConfusionMatrixDisplay,
#     accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# )

# # ---------- output dirs ----------
# PLOTS_DIR = Path("ml_results/plots")
# TABLES_DIR = Path("ml_results/tables")
# PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# TABLES_DIR.mkdir(parents=True, exist_ok=True)

# # ---------- scenario labels ----------
# SCENARIO_LABELS = {
#     1: 'S1: BERT (Text Only)',
#     2: 'S2: n=49 (ALL_40)',
#     3: 'S5: n=25 (full)',
#     4: 'S4: n=14 (mixed)',
#     5: 'S3: n=6 (core)',
# }

# # ---------- if not already defined ----------
# # MODEL_COLORS should already exist in your notebook.
# # If not, uncomment and use this fallback:
# # MODEL_COLORS = {
# #     'BERT': '#1565C0', 'GRU': '#6A1B9A', 'LSTM': '#00695C',
# #     'Logistic_Regression': '#E65100', 'Ridge_Classifier': '#BF360C',
# #     'SVM_RBF': '#4E342E', 'SVM_Calibrated': '#37474F', 'LinearSVC': '#263238',
# #     'Random_Forest': '#2E7D32', 'Extra_Trees': '#558B2F',
# #     'XGBoost': '#F9A825', 'LightGBM': '#00838F',
# #     'KNN': '#6D4C41', 'Decision_Tree': '#8D6E63',
# #     'SGD': '#546E7A', 'Naive_Bayes': '#7B1FA2',
# # }

# # ============================================================
# # 1) Find top-1 model per scenario by ROC-AUC
# # ============================================================
# # best_per_scenario = (
# #     results_df.sort_values(['scenario', 'roc_auc'], ascending=[True, False])
# #               .groupby('scenario', as_index=False)
# #               .head(1)
# #               .sort_values('scenario')
# #               .reset_index(drop=True)
# # )
# best_per_scenario = (
#     results_df.sort_values(['scenario', 'f1', 'roc_auc'],
#                            ascending=[True, False, False])
#               .groupby('scenario', as_index=False)
#               .head(1)
#               .sort_values('scenario')
#               .reset_index(drop=True)
# )

# print("\nTop-1 model per scenario:")
# print(best_per_scenario[['scenario', 'model', 'feature_set', 'roc_auc', 'f1']].to_string(index=False))

# # ============================================================
# # 2) Helper to fetch the full prediction object from ALL_RESULTS
# # ============================================================
# def fetch_result(row):
#     s = int(row['scenario'])
#     for r in ALL_RESULTS:
#         if int(r.get('scenario', -999)) == s and \
#            r.get('model') == row['model'] and \
#            r.get('feature_set') == row['feature_set']:
#             return r
#     # fallback if scenario key is missing for any reason
#     for r in ALL_RESULTS:
#         if r.get('model') == row['model'] and r.get('feature_set') == row['feature_set']:
#             return r
#     raise KeyError(f"Could not find result for scenario={s}, model={row['model']}, feature_set={row['feature_set']}")

# # ============================================================
# # 3) Build summary + objects for plotting
# # ============================================================
# summary_rows = []
# plot_items = []

# for _, row in best_per_scenario.iterrows():
#     res = fetch_result(row)

#     y_true = np.asarray(res['y_test'])
#     y_pred = np.asarray(res['y_pred'])
#     y_prob = np.asarray(res['y_prob'])

#     # curves
#     fpr, tpr, _ = roc_curve(y_true, y_prob)
#     roc_auc_v = auc(fpr, tpr)

#     prec, rec, _ = precision_recall_curve(y_true, y_prob)
#     ap = average_precision_score(y_true, y_prob)

#     cm = confusion_matrix(y_true, y_pred)

#     # metrics
#     acc = accuracy_score(y_true, y_pred)
#     pre = precision_score(y_true, y_pred, zero_division=0)
#     rec_s = recall_score(y_true, y_pred, zero_division=0)
#     f1 = f1_score(y_true, y_pred, zero_division=0)
#     roc_auc_prob = roc_auc_score(y_true, y_prob)

#     summary_rows.append({
#         'scenario': int(row['scenario']),
#         'scenario_label': SCENARIO_LABELS.get(int(row['scenario']), f"Scenario {int(row['scenario'])}"),
#         'model': row['model'],
#         'feature_set': row['feature_set'],
#         'accuracy': round(acc, 4),
#         'precision': round(pre, 4),
#         'recall': round(rec_s, 4),
#         'f1': round(f1, 4),
#         'roc_auc': round(roc_auc_prob, 4),
#         'avg_precision': round(ap, 4),
#         'train_time_s': float(res.get('train_time_s', np.nan)),
#         'best_params': res.get('best_params', '')
#     })

#     plot_items.append({
#         'row': row,
#         'res': res,
#         'y_true': y_true,
#         'y_pred': y_pred,
#         'y_prob': y_prob,
#         'fpr': fpr,
#         'tpr': tpr,
#         'roc_auc': roc_auc_v,
#         'prec': prec,
#         'rec': rec,
#         'ap': ap,
#         'cm': cm,
#         'metrics': {
#             'accuracy': acc,
#             'precision': pre,
#             'recall': rec_s,
#             'f1': f1,
#             'roc_auc': roc_auc_prob,
#         }
#     })

# top1_summary_df = pd.DataFrame(summary_rows).sort_values('scenario')
# top1_summary_df.to_csv(TABLES_DIR / "top1_models_across_5_scenarios.csv", index=False)

# print("\nTop-1 summary saved to:")
# print(TABLES_DIR / "top1_models_across_5_scenarios.csv")
# print(top1_summary_df.to_string(index=False))

# # ============================================================
# # 4) Combined dashboard: ROC + PR + Confusion Matrix
# #    one row per scenario, three columns
# # ============================================================
# n = len(plot_items)
# fig, axes = plt.subplots(nrows=n, ncols=3, figsize=(22, 4.6 * n))
# if n == 1:
#     axes = np.array([axes])

# for i, item in enumerate(plot_items):
#     row = item['row']
#     sc = int(row['scenario'])
#     model = row['model']
#     fs = row['feature_set']
#     color = MODEL_COLORS.get(model, "#1f77b4")
#     label = f"{SCENARIO_LABELS.get(sc, sc)}\n{model} [{fs}]"

#     # --- ROC ---
#     ax = axes[i, 0]
#     ax.plot(item['fpr'], item['tpr'], color=color, lw=2.5,
#             label=f"AUC = {item['roc_auc']:.3f}")
#     ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
#     ax.set_title(f"ROC Curve\n{label}", fontweight='bold', fontsize=10)
#     ax.set_xlabel("False Positive Rate")
#     ax.set_ylabel("True Positive Rate")
#     ax.grid(alpha=0.2)
#     ax.legend(loc="lower right", fontsize=8)

#     # --- PR ---
#     ax = axes[i, 1]
#     baseline = item['y_true'].mean()
#     ax.plot(item['rec'], item['prec'], color=color, lw=2.5,
#             label=f"AP = {item['ap']:.3f}")
#     ax.axhline(baseline, color='gray', ls='--', lw=1.2,
#                label=f"Baseline = {baseline:.3f}")
#     ax.set_title(f"Precision-Recall\n{label}", fontweight='bold', fontsize=10)
#     ax.set_xlabel("Recall")
#     ax.set_ylabel("Precision")
#     ax.grid(alpha=0.2)
#     ax.legend(loc="lower left", fontsize=8)

#     # --- Confusion Matrix ---
#     ax = axes[i, 2]
#     disp = ConfusionMatrixDisplay(item['cm'], display_labels=['Human', 'AI'])
#     disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
#     m = item['metrics']
#     ax.set_title(
#         f"Confusion Matrix\n{label}\n"
#         f"Acc={m['accuracy']:.3f}  Prec={m['precision']:.3f}  "
#         f"Rec={m['recall']:.3f}  F1={m['f1']:.3f}",
#         fontweight='bold', fontsize=10
#     )
#     ax.set_xlabel("Predicted")
#     ax.set_ylabel("True")

# plt.tight_layout(rect=[0, 0, 1, 0.99])
# dashboard_path = PLOTS_DIR / "top1_models_dashboard_roc_pr_cm.png"
# plt.savefig(dashboard_path, dpi=220, bbox_inches='tight')
# plt.close()

# print(f"\nSaved combined dashboard -> {dashboard_path}")

# # ============================================================
# # 5) Optional: one overlay ROC plot for the 5 top models
# # ============================================================
# fig, ax = plt.subplots(figsize=(9, 7))
# for item in plot_items:
#     row = item['row']
#     sc = int(row['scenario'])
#     color = MODEL_COLORS.get(row['model'], "#1f77b4")
#     ax.plot(item['fpr'], item['tpr'], lw=2.2, color=color,
#             label=f"{SCENARIO_LABELS.get(sc, sc)} | {row['model']} | AUC={item['roc_auc']:.3f}")

# ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
# ax.set_title("Combined ROC Curves — Top-1 Model per Scenario", fontweight='bold')
# ax.set_xlabel("False Positive Rate")
# ax.set_ylabel("True Positive Rate")
# ax.grid(alpha=0.2)
# ax.legend(fontsize=8, loc='lower right')
# plt.tight_layout()
# roc_overlay_path = PLOTS_DIR / "top1_models_combined_roc.png"
# plt.savefig(roc_overlay_path, dpi=220, bbox_inches='tight')
# plt.close()

# print(f"Saved ROC overlay -> {roc_overlay_path}")

# # ============================================================
# # 6) Optional: one overlay PR plot for the 5 top models
# # ============================================================
# fig, ax = plt.subplots(figsize=(9, 7))
# for item in plot_items:
#     row = item['row']
#     sc = int(row['scenario'])
#     color = MODEL_COLORS.get(row['model'], "#1f77b4")
#     baseline = item['y_true'].mean()
#     ax.plot(item['rec'], item['prec'], lw=2.2, color=color,
#             label=f"{SCENARIO_LABELS.get(sc, sc)} | {row['model']} | AP={item['ap']:.3f}")
#     ax.axhline(baseline, color=color, ls='--', alpha=0.12)

# ax.set_title("Combined Precision-Recall Curves — Top-1 Model per Scenario", fontweight='bold')
# ax.set_xlabel("Recall")
# ax.set_ylabel("Precision")
# ax.grid(alpha=0.2)
# ax.legend(fontsize=8, loc='lower left')
# plt.tight_layout()
# pr_overlay_path = PLOTS_DIR / "top1_models_combined_pr.png"
# plt.savefig(pr_overlay_path, dpi=220, bbox_inches='tight')
# plt.close()

# print(f"Saved PR overlay -> {pr_overlay_path}")

# # ============================================================
# # Done
# # ============================================================
# print("\n✅ Finished generating combined ROC, PR, and confusion matrix plots for top-1 models across all 5 scenarios.")

# ============================================================
# TOP-1 MODELS ACROSS ALL 5 SCENARIOS
# Combined: ROC curve + PR curve + Confusion Matrix
# Colours are assigned PER SCENARIO (not per model)
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# ── Output dirs ───────────────────────────────────────────────
PLOTS_DIR  = Path("ml_results/plots")
TABLES_DIR = Path("ml_results/tables")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── Scenario labels ───────────────────────────────────────────
SCENARIO_LABELS = {
    1: 'S1: BERT (Text Only)',
    2: 'S2: n=49 (All_40)',
    3: 'S3: n=25 (full)',
    4: 'S4: n=14 (mixed)',
    5: 'S5: n=6 (core)',
}

# ── ONE colour per scenario ───────────────────────────────────
SCENARIO_COLORS = {
    1: '#1565C0',   # deep blue      → BERT / text-only
    2: '#2E7D32',   # deep green     → All 40
    3: '#E65100',   # deep orange    → 25 features
    4: '#6A1B9A',   # deep purple    → 14 features
    5: '#AD1457',   # deep pink/red  → 6 core features
}

# ── Scenario-aware cmaps for confusion matrices ───────────────
SCENARIO_CMAPS = {
    1: 'Blues',
    2: 'Greens',
    3: 'Oranges',
    4: 'Purples',
    5: 'RdPu',
}

# ============================================================
# 1) Find top-1 model per scenario  (F1 primary, AUC secondary)
# ============================================================
best_per_scenario = (
    results_df
    .sort_values(['scenario', 'f1', 'roc_auc'],
                 ascending=[True, False, False])
    .groupby('scenario', as_index=False)
    .head(1)
    .sort_values('scenario')
    .reset_index(drop=True)
)

print("\nTop-1 model per scenario (F1 → AUC):")
print(best_per_scenario[
    ['scenario', 'model', 'feature_set', 'f1', 'roc_auc']
].to_string(index=False))

# ============================================================
# 2) Helper: fetch full prediction dict from ALL_RESULTS
# ============================================================
def fetch_result(row):
    s = int(row['scenario'])
    for r in ALL_RESULTS:
        if (int(r.get('scenario', -999)) == s and
                r.get('model')       == row['model'] and
                r.get('feature_set') == row['feature_set']):
            return r
    for r in ALL_RESULTS:          # fallback
        if (r.get('model')       == row['model'] and
                r.get('feature_set') == row['feature_set']):
            return r
    raise KeyError(
        f"Cannot find scenario={s}, "
        f"model={row['model']}, fs={row['feature_set']}")

# ============================================================
# 3) Build summary table + plot_items
# ============================================================
summary_rows = []
plot_items   = []

for _, row in best_per_scenario.iterrows():
    res    = fetch_result(row)
    y_true = np.asarray(res['y_test'])
    y_pred = np.asarray(res['y_pred'])
    y_prob = np.asarray(res['y_prob'])

    fpr, tpr, _       = roc_curve(y_true, y_prob)
    roc_auc_v         = auc(fpr, tpr)
    prec_c, rec_c, _  = precision_recall_curve(y_true, y_prob)
    ap                = average_precision_score(y_true, y_prob)
    cm                = confusion_matrix(y_true, y_pred)

    acc   = accuracy_score(y_true, y_pred)
    pre   = precision_score(y_true, y_pred, zero_division=0)
    rec_s = recall_score(y_true, y_pred,    zero_division=0)
    f1_s  = f1_score(y_true, y_pred,        zero_division=0)
    rauc  = roc_auc_score(y_true, y_prob)

    sc = int(row['scenario'])

    summary_rows.append({
        'scenario':       sc,
        'scenario_label': SCENARIO_LABELS.get(sc, f"S{sc}"),
        'model':          row['model'],
        'feature_set':    row['feature_set'],
        'accuracy':  round(acc,   4),
        'precision': round(pre,   4),
        'recall':    round(rec_s, 4),
        'f1':        round(f1_s,  4),
        'roc_auc':   round(rauc,  4),
        'avg_precision': round(ap, 4),
        'train_time_s': float(res.get('train_time_s', np.nan)),
        'best_params':  res.get('best_params', ''),
    })

    plot_items.append({
        'row':     row,
        'sc':      sc,
        'color':   SCENARIO_COLORS.get(sc, '#333333'),
        'cmap':    SCENARIO_CMAPS.get(sc,  'Blues'),
        'label':   SCENARIO_LABELS.get(sc, f"S{sc}"),
        'y_true':  y_true,
        'y_pred':  y_pred,
        'y_prob':  y_prob,
        'fpr':     fpr,
        'tpr':     tpr,
        'roc_auc': roc_auc_v,
        'prec_c':  prec_c,
        'rec_c':   rec_c,
        'ap':      ap,
        'cm':      cm,
        'metrics': {
            'accuracy':  acc,
            'precision': pre,
            'recall':    rec_s,
            'f1':        f1_s,
            'roc_auc':   rauc,
        },
    })

top1_df = (pd.DataFrame(summary_rows)
           .sort_values('scenario')
           .reset_index(drop=True))
csv_path = TABLES_DIR / "top1_f1_models_across_5_scenarios.csv"
top1_df.to_csv(csv_path, index=False)
print(f"\nSummary saved → {csv_path}")
print(top1_df.to_string(index=False))

# ============================================================
# 4) DASHBOARD  5 rows × 3 cols
#    ROC | PR | Confusion Matrix   — scenario-coloured
# ============================================================
n   = len(plot_items)
fig, axes = plt.subplots(nrows=n, ncols=3,
                          figsize=(22, 4.8 * n))
if n == 1:
    axes = np.array([axes])

for i, item in enumerate(plot_items):
    row   = item['row']
    sc    = item['sc']
    color = item['color']
    cmap  = item['cmap']
    m     = item['metrics']
    s_lbl = item['label']
    mdl   = row['model'].replace('_', ' ')
    fs    = row['feature_set']
    title_tag = f"{s_lbl}\n{mdl}  [{fs}]"

    # ── ROC ───────────────────────────────────────────────
    ax = axes[i, 0]
    ax.plot(item['fpr'], item['tpr'],
            color=color, lw=2.8,
            label=f"AUC = {item['roc_auc']:.3f}")
    ax.fill_between(item['fpr'], item['tpr'],
                    alpha=0.10, color=color)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.35)
    ax.set_title(f"ROC Curve\n{title_tag}",
                 fontweight='bold', fontsize=10, color=color)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.18)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # annotation box
    ax.text(0.60, 0.10,
            f"F1  = {m['f1']:.3f}\n"
            f"Acc = {m['accuracy']:.3f}",
            transform=ax.transAxes,
            fontsize=8.5, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white',
                      edgecolor=color,
                      linewidth=1.2))

    # ── PR ────────────────────────────────────────────────
    ax = axes[i, 1]
    baseline = item['y_true'].mean()
    ax.plot(item['rec_c'], item['prec_c'],
            color=color, lw=2.8,
            label=f"AP = {item['ap']:.3f}")
    ax.fill_between(item['rec_c'], item['prec_c'],
                    alpha=0.10, color=color)
    ax.axhline(baseline, color='#555555',
               ls='--', lw=1.3,
               label=f"Baseline = {baseline:.3f}")
    ax.set_title(f"Precision-Recall\n{title_tag}",
                 fontweight='bold', fontsize=10, color=color)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left", fontsize=9)
    ax.grid(alpha=0.18)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.text(0.02, 0.14,
            f"Prec = {m['precision']:.3f}\n"
            f"Rec  = {m['recall']:.3f}",
            transform=ax.transAxes,
            fontsize=8.5, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white',
                      edgecolor=color,
                      linewidth=1.2))

    # ── Confusion Matrix ──────────────────────────────────
    ax = axes[i, 2]
    disp = ConfusionMatrixDisplay(
        confusion_matrix=item['cm'],
        display_labels=['Human', 'AI'])
    disp.plot(ax=ax, cmap=cmap,
              colorbar=False, values_format='d')
    for txt in ax.texts:
        txt.set_fontsize(15)
        txt.set_fontweight('bold')
    ax.set_title(
        f"Confusion Matrix\n{title_tag}\n"
        f"Acc={m['accuracy']:.3f}  "
        f"Prec={m['precision']:.3f}  "
        f"Rec={m['recall']:.3f}  "
        f"F1={m['f1']:.3f}",
        fontweight='bold', fontsize=10, color=color)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

fig.suptitle(
    "Top-1 Model per Scenario — Ranked by F1 Score\n"
    "ROC Curve  |  Precision-Recall  |  Confusion Matrix\n",
    fontsize=14, fontweight='bold', y=1.005)

plt.tight_layout()
dash_path = PLOTS_DIR / "top1_f1_dashboard_roc_pr_cm.png"
plt.savefig(dash_path, dpi=220,
            bbox_inches='tight', facecolor='white')
plt.close()
print(f"\n✅ Dashboard saved → {dash_path}")

# ============================================================
# 5) OVERLAY ROC — all 5 scenarios on one axes
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7.5))

for item in plot_items:
    sc    = item['sc']
    color = item['color']
    m     = item['metrics']
    mdl   = item['row']['model'].replace('_', ' ')
    lbl   = (f"{item['label']}  |  {mdl}  |  "
             f"AUC={item['roc_auc']:.3f}  F1={m['f1']:.3f}")
    ax.plot(item['fpr'], item['tpr'],
            lw=2.8, color=color, label=lbl)
    ax.fill_between(item['fpr'], item['tpr'],
                    alpha=0.07, color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=1,
        alpha=0.35, label='Random classifier')
ax.set_title(
    "Combined ROC Curves\n"
    "Top-1 Model per Scenario",
    fontweight='bold', fontsize=13)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.legend(fontsize=9, loc='lower right',
          framealpha=0.9)
ax.grid(alpha=0.18)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
roc_path = PLOTS_DIR / "top1_f1_combined_roc.png"
plt.savefig(roc_path, dpi=220,
            bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Combined ROC saved → {roc_path}")

# ============================================================
# 6) OVERLAY PR — all 5 scenarios on one axes
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7.5))

for item in plot_items:
    sc    = item['sc']
    color = item['color']
    m     = item['metrics']
    mdl   = item['row']['model'].replace('_', ' ')
    lbl   = (f"{item['label']}  |  {mdl}  |  "
             f"AP={item['ap']:.3f}  F1={m['f1']:.3f}")
    ax.plot(item['rec_c'], item['prec_c'],
            lw=2.8, color=color, label=lbl)
    ax.fill_between(item['rec_c'], item['prec_c'],
                    alpha=0.07, color=color)

ax.set_title(
    "Combined Precision-Recall Curves\n"
    "Top-1 Model per Scenario",
    fontweight='bold', fontsize=13)
ax.set_xlabel("Recall",    fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.legend(fontsize=9, loc='lower left',
          framealpha=0.9)
ax.grid(alpha=0.18)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
pr_path = PLOTS_DIR / "top1_f1_combined_pr.png"
plt.savefig(pr_path, dpi=220,
            bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Combined PR  saved → {pr_path}")

# ============================================================
# 7) STANDALONE CM — 1 × 5 horizontal strip
# ============================================================
fig, axes = plt.subplots(nrows=1, ncols=n,
                          figsize=(5.8 * n, 7.2))
if n == 1:
    axes = [axes]

for i, item in enumerate(plot_items):
    ax    = axes[i]
    sc    = item['sc']
    color = item['color']
    cmap  = item['cmap']
    m     = item['metrics']
    mdl   = item['row']['model'].replace('_', ' ')
    fs    = item['row']['feature_set']

    disp = ConfusionMatrixDisplay(
        confusion_matrix=item['cm'],
        display_labels=['Human', 'AI'])
    disp.plot(ax=ax, cmap=cmap,
              colorbar=False, values_format='d')
    for txt in ax.texts:
        txt.set_fontsize(18)
        txt.set_fontweight('bold')

    # coloured badge title
    ax.set_title(
        item['label'],
        fontsize=11, fontweight='bold',
        color='white', pad=8,
        bbox=dict(boxstyle='round,pad=0.4',
                  facecolor=color,
                  edgecolor='none', alpha=0.93))

    ax.text(0.5, -0.08,
            f"{mdl}  [{fs}]",
            ha='center', va='top',
            transform=ax.transAxes,
            fontsize=9, fontstyle='italic',
            color='#333333')

    ax.text(0.5, -0.22,
            f"Acc={m['accuracy']:.3f}   "
            f"Prec={m['precision']:.3f}\n"
            f"Rec={m['recall']:.3f}    "
            f"F1={m['f1']:.3f}    "
            f"AUC={m['roc_auc']:.3f}",
            ha='center', va='top',
            transform=ax.transAxes,
            fontsize=8.5, family='monospace',
            color='#111111',
            bbox=dict(boxstyle='round,pad=0.35',
                      facecolor='#F7F7F7',
                      edgecolor=color,
                      linewidth=1.0))

    ax.set_xlabel("Predicted Label", fontsize=10)
    ax.set_ylabel("True Label",      fontsize=10)

fig.suptitle(
    "Confusion Matrices — Top-1 Model per Scenario (ranked by F1)\n"
    "Evaluated on External Test Set",
    fontsize=13, fontweight='bold', y=1.07)
plt.tight_layout(rect=[0, 0.10, 1, 1])
cm1_path = PLOTS_DIR / "top1_f1_cm_1x5.png"
plt.savefig(cm1_path, dpi=220,
            bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ CM strip  (1×5) saved → {cm1_path}")

# ============================================================
# 8) STANDALONE CM — 2 × 3 grid
# ============================================================
fig2, axes2 = plt.subplots(nrows=2, ncols=3,
                             figsize=(18, 12))
flat = axes2.flatten()

for i, item in enumerate(plot_items):
    ax    = flat[i]
    sc    = item['sc']
    color = item['color']
    cmap  = item['cmap']
    m     = item['metrics']
    mdl   = item['row']['model'].replace('_', ' ')
    fs    = item['row']['feature_set']

    disp = ConfusionMatrixDisplay(
        confusion_matrix=item['cm'],
        display_labels=['Human', 'AI'])
    disp.plot(ax=ax, cmap=cmap,
              colorbar=False, values_format='d')
    for txt in ax.texts:
        txt.set_fontsize(20)
        txt.set_fontweight('bold')

    ax.set_title(
        f"{item['label']}\n"
        f"{mdl}  ·  [{fs}]",
        fontsize=11, fontweight='bold',
        color=color, pad=10)

    ax.text(0.5, -0.13,
            f"Acc={m['accuracy']:.3f}  "
            f"Prec={m['precision']:.3f}  "
            f"Rec={m['recall']:.3f}  "
            f"F1={m['f1']:.3f}  "
            f"AUC={m['roc_auc']:.3f}",
            ha='center', va='top',
            transform=ax.transAxes,
            fontsize=9, family='monospace',
            color='#111111',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='#FAFAFA',
                      edgecolor=color,
                      linewidth=1.0))

    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label",      fontsize=11)

# ── Hide spare cell ───────────────────────────────────────────
flat[n].axis('off')

fig2.suptitle(
    "Confusion Matrices — Top-1 Model per Scenario (ranked by F1)\n"
    "Evaluated on External Test Set",
    fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
cm2_path = PLOTS_DIR / "top1_f1_cm_2x3.png"
plt.savefig(cm2_path, dpi=220,
            bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ CM grid   (2×3) saved → {cm2_path}")

# ============================================================
# Summary
# ============================================================
print("\n" + "="*65)
print("SCENARIO COLOUR LEGEND")
print("="*65)
for sc, col in SCENARIO_COLORS.items():
    lbl = SCENARIO_LABELS.get(sc, f"S{sc}")
    print(f"  S{sc}  {col}   {lbl}")

print("\nFILES SAVED")
print("="*65)
for p in [dash_path, roc_path, pr_path, cm1_path, cm2_path, csv_path]:
    kb = Path(p).stat().st_size / 1024
    print(f"  {Path(p).name:<52}  {kb:>7.1f} KB")
print("="*65)
print("\n✅ All scenario-coloured plots complete.")


Top-1 model per scenario (F1 → AUC):
 scenario   model feature_set     f1  roc_auc
        1    BERT   Text_Only 0.9650   0.9942
        2    LSTM      All_40 0.9428   0.9846
        3     GRU    Case1_25 0.9296   0.9818
        4     GRU    Case2_14 0.8996   0.9629
        5 XGBoost     Case3_6 0.8433   0.9278

Summary saved → ml_results\tables\top1_f1_models_across_5_scenarios.csv
 scenario       scenario_label   model feature_set  accuracy  precision  recall     f1  roc_auc  avg_precision  train_time_s                                                                                                                      best_params
        1 S1: BERT (Text Only)    BERT   Text_Only    0.9646     0.9554  0.9747 0.9650   0.9942         0.9954       7772.14                                                                                            lr=2e-05,batch=4,maxlen=128,fp16=True
        2    S2: n=49 (All_40)    LSTM      All_40    0.9419     0.9289  0.9571 0.9428   0.9846         0

In [26]:
# ============================================================
# SEPARATE PNG: CONFUSION MATRICES FOR TOP-1 MODELS
# ============================================================

from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
axes = axes.flatten()

for i, item in enumerate(plot_items):
    row = item['row']
    sc = int(row['scenario'])
    model = row['model']
    fs = row['feature_set']
    m = item['metrics']
    color = MODEL_COLORS.get(model, "#1f77b4")

    ax = axes[i]

    disp = ConfusionMatrixDisplay(
        confusion_matrix=item['cm'],
        display_labels=['Human', 'AI']
    )
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')

    ax.set_title(
        f"{SCENARIO_LABELS.get(sc, sc)}\n"
        f"{model} [{fs}]\n"
        f"Acc={m['accuracy']:.3f}  Prec={m['precision']:.3f}  "
        f"Rec={m['recall']:.3f}  F1={m['f1']:.3f}",
        fontsize=10, fontweight='bold'
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

# turn off any unused axes
for j in range(len(plot_items), len(axes)):
    axes[j].axis('off')

plt.suptitle(
    "Confusion Matrices — Top-1 Model per Scenario",
    fontsize=14, fontweight='bold'
)
plt.tight_layout(rect=[0, 0, 1, 0.96])

cm_path = PLOTS_DIR / "top1_models_confusion_matrices.png"
plt.savefig(cm_path, dpi=220, bbox_inches='tight')
plt.close()

print(f"Saved confusion matrices -> {cm_path}")

Saved confusion matrices -> ml_results\plots\top1_models_confusion_matrices.png


In [ ]:
# ============================================================
# OPTIONAL: NORMALIZED CONFUSION MATRICES
# ============================================================

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
axes = axes.flatten()

for i, item in enumerate(plot_items):
    row = item['row']
    sc = int(row['scenario'])
    model = row['model']
    fs = row['feature_set']
    ax = axes[i]

    cm = item['cm'].astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_norm,
        display_labels=['Human', 'AI']
    )
    disp.plot(ax=ax, cmap='Greens', colorbar=False, values_format='.2f')

    ax.set_title(
        f"{SCENARIO_LABELS.get(sc, sc)}\n{model} [{fs}]",
        fontsize=10, fontweight='bold'
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

for j in range(len(plot_items), len(axes)):
    axes[j].axis('off')

plt.suptitle(
    "Normalized Confusion Matrices — Top-1 Model per Scenario",
    fontsize=14, fontweight='bold'
)
plt.tight_layout(rect=[0, 0, 1, 0.96])

cm_norm_path = PLOTS_DIR / "top1_models_confusion_matrices_normalized.png"
plt.savefig(cm_norm_path, dpi=220, bbox_inches='tight')
plt.close()

print(f"Saved normalized confusion matrices -> {cm_norm_path}")

Saved normalized confusion matrices -> ml_results\plots\top1_models_confusion_matrices_normalized.png


In [ ]:
# ============================================================
# GENERATE LaTeX TABLE FROM results_df
# Mirrors the structure of tab:performance_hierarchy
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

TABLES_DIR = Path("ml_results/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── 1) Scenario → feature-set mapping ────────────────────────
SCENARIO_FS = {
    1: 'Text_Only',
    2: 'All_40',
    3: 'Case3_25',
    4: 'Case2_14',
    5: 'Case1_6',
}

# ── 2) Display names for models (order = table row order) ────
MODEL_DISPLAY = {
    # Deep Learning
    'LSTM':                'LSTM',
    'GRU':                 'GRU',
    # Ensemble & Boosting
    'LightGBM':            'LightGBM',
    'XGBoost':             'XGBoost',
    'Extra_Trees':         'Extra Trees',
    'Random_Forest':       'Random Forest',
    # Linear & SVM
    'SVM_RBF':             'SVM (RBF Kernel)',
    'SVM_Calibrated':      'SVM (Calibrated)',
    'LinearSVC':           'Linear SVC',
    'Logistic_Regression': 'Logistic Regression',
    'Ridge_Classifier':    'Ridge Classifier',
    'SGD':                 'SGD Classifier',
    # Instance-based & Probabilistic
    'KNN':                 'K-Nearest Neighbors',
    'Decision_Tree':       'Decision Tree',
    'Naive_Bayes':         'Naive Bayes',
}

# ── 3) Group headings and which models fall under each ───────
GROUPS = [
    (r'\textit{Deep Learning (Hybrid Paradigm)}',
     ['LSTM', 'GRU']),

    (r'\textit{Ensemble \& Boosting (Glass-Box Paradigm)}',
     ['LightGBM', 'XGBoost', 'Extra_Trees', 'Random_Forest']),

    (r'\textit{Linear Models \& Support Vector Machines (Glass-Box Paradigm)}',
     ['SVM_RBF', 'SVM_Calibrated', 'LinearSVC',
      'Logistic_Regression', 'Ridge_Classifier', 'SGD']),

    (r'\textit{Instance-based \& Probabilistic Classifiers}',
     ['KNN', 'Decision_Tree', 'Naive_Bayes']),
]

# ── 4) Pull BERT (Scenario 1) metrics ────────────────────────
bert_row = results_df[
    (results_df['model'] == 'BERT') &
    (results_df['feature_set'] == 'Text_Only')
]

if bert_row.empty:
    bert_acc = bert_f1 = bert_auc = float('nan')
else:
    bert_acc = float(bert_row['accuracy'].iloc[0])
    bert_f1  = float(bert_row['f1'].iloc[0])
    bert_auc = float(bert_row['roc_auc'].iloc[0])

print(f"BERT baseline  Acc={bert_acc:.3f}  "
      f"F1={bert_f1:.3f}  AUC={bert_auc:.3f}")

# ── 5) Helper: look up one cell from results_df ──────────────
def get_metrics(model_key, scenario_num):
    """
    Return (accuracy, f1, roc_auc) for a given model
    and scenario number.  Returns ('--','--','--') if missing.
    """
    fs  = SCENARIO_FS.get(scenario_num, '')
    sub = results_df[
        (results_df['model']       == model_key) &
        (results_df['feature_set'] == fs)
    ]
    if sub.empty:
        return ('--', '--', '--')
    r = sub.iloc[0]
    return (
        f"{float(r['accuracy']):.3f}",
        f"{float(r['f1']):.3f}",
        f"{float(r['roc_auc']):.3f}",
    )

# ── 6) Build the LaTeX string ─────────────────────────────────
def build_latex_table():
    lines = []
    a = lines.append   # shorthand

    # ── Table header ─────────────────────────────────────
    a(r'\begin{table*}[htbp!]')
    a(r'\centering')
    a(r'\caption{Comprehensive performance comparison across '
      r'experimental scenarios. The table illustrates the monotonic '
      r'degradation in classification metrics (Accuracy, '
      r'$F_1$-score, and ROC-AUC) as the feature space is reduced '
      r'from the full stylometric taxonomy (All\_40 (n=49)) to the '
      r'minimal core set (n = 6). The black-box BERT baseline '
      r'(Scenario 1) is provided for reference.}')
    a(r'\label{tab:performance_hierarchy}')
    a(r'\resizebox{\textwidth}{!}{%')
    a(r'\renewcommand{\arraystretch}{1.2}')
    a(r'\begin{tabular}{@{}lcccccccccccc@{}}')
    a(r'\toprule')

    # ── BERT baseline row ─────────────────────────────────
    a(r'\multicolumn{13}{@{}l}{\textbf{Baseline Reference: '
      r'Scenario 1 (Text-Only Black-Box)}} \\')
    a(r'\midrule')
    a(
        r'BERT & \multicolumn{12}{c}{'
        r'\textbf{Accuracy:} ' + f'{bert_acc:.3f}' +
        r' \quad | \quad \textbf{F1-Score:} ' + f'{bert_f1:.3f}' +
        r' \quad | \quad \textbf{ROC-AUC:} ' + f'{bert_auc:.3f}' +
        r'} \\'
    )
    a(r'\midrule')
    a(r'\midrule')

    # ── Column headers ────────────────────────────────────
    a(r'\multirow{2}{*}{\textbf{Model Algorithm}} '
      r'& \multicolumn{3}{c}{\textbf{Scenario 2: All\_40 (n=49)}} '
      r'& \multicolumn{3}{c}{\textbf{Scenario 3: n=25}} '
      r'& \multicolumn{3}{c}{\textbf{Scenario 4: n=14}} '
      r'& \multicolumn{3}{c}{\textbf{Scenario 5: n=6}} \\')
    a(r'\cmidrule(lr){2-4} \cmidrule(lr){5-7} '
      r'\cmidrule(lr){8-10} \cmidrule(lr){11-13}')
    a(r'& \textbf{Acc} & \textbf{F1} & \textbf{AUC} '
      r'& \textbf{Acc} & \textbf{F1} & \textbf{AUC} '
      r'& \textbf{Acc} & \textbf{F1} & \textbf{AUC} '
      r'& \textbf{Acc} & \textbf{F1} & \textbf{AUC} \\')
    a(r'\midrule')

    # ── Model rows grouped ────────────────────────────────
    for group_heading, model_keys in GROUPS:
        a(r'\multicolumn{13}{@{}l}{' + group_heading + r'} \\')
        a(r'\midrule')

        for mk in model_keys:
            display = MODEL_DISPLAY.get(mk, mk.replace('_', ' '))

            # fetch metrics for scenarios 2 → 5
            s2 = get_metrics(mk, 2)
            s3 = get_metrics(mk, 3)
            s4 = get_metrics(mk, 4)
            s5 = get_metrics(mk, 5)

            row_str = (
                f'{display:<22}'
                f' & {s2[0]} & {s2[1]} & {s2[2]}'
                f' & {s3[0]} & {s3[1]} & {s3[2]}'
                f' & {s4[0]} & {s4[1]} & {s4[2]}'
                f' & {s5[0]} & {s5[1]} & {s5[2]}'
                r' \\'
            )
            a(row_str)

        a(r'\midrule')

    # ── Footer ────────────────────────────────────────────
    # Remove the last \midrule and replace with \bottomrule
    lines[-1] = r'\bottomrule'

    a(r'\end{tabular}}')
    a(r'\end{table*}')

    return '\n'.join(lines)


latex_str = build_latex_table()

# ── 7) Print to console ───────────────────────────────────────
print("\n" + "="*70)
print("GENERATED LaTeX TABLE")
print("="*70)
print(latex_str)
print("="*70)

# ── 8) Save to file ───────────────────────────────────────────
tex_path = TABLES_DIR / "performance_hierarchy_table.tex"
tex_path.write_text(latex_str, encoding='utf-8')
print(f"\n✅ LaTeX table saved → {tex_path}")

# ── 9) Quick sanity-check print (readable grid) ───────────────
print("\n── Sanity check (raw values) ──")
header = (f"{'Model':<25} "
          f"{'S2_Acc':>7} {'S2_F1':>6} {'S2_AUC':>7}  "
          f"{'S3_Acc':>7} {'S3_F1':>6} {'S3_AUC':>7}  "
          f"{'S4_Acc':>7} {'S4_F1':>6} {'S4_AUC':>7}  "
          f"{'S5_Acc':>7} {'S5_F1':>6} {'S5_AUC':>7}")
print(header)
print("-" * len(header))

for _, model_keys in GROUPS:
    for mk in model_keys:
        display = MODEL_DISPLAY.get(mk, mk)
        s2 = get_metrics(mk, 2)
        s3 = get_metrics(mk, 3)
        s4 = get_metrics(mk, 4)
        s5 = get_metrics(mk, 5)
        print(
            f"{display:<25} "
            f"{s2[0]:>7} {s2[1]:>6} {s2[2]:>7}  "
            f"{s3[0]:>7} {s3[1]:>6} {s3[2]:>7}  "
            f"{s4[0]:>7} {s4[1]:>6} {s4[2]:>7}  "
            f"{s5[0]:>7} {s5[1]:>6} {s5[2]:>7}"
        )

BERT baseline  Acc=0.965  F1=0.965  AUC=0.994

GENERATED LaTeX TABLE
\begin{table*}[htbp!]
\centering
\caption{Comprehensive performance comparison across experimental scenarios. The table illustrates the monotonic degradation in classification metrics (Accuracy, $F_1$-score, and ROC-AUC) as the feature space is reduced from the full stylometric taxonomy (All\_40 (n=49)) to the minimal core set (n = 6). The black-box BERT baseline (Scenario 1) is provided for reference.}
\label{tab:performance_hierarchy}
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2}
\begin{tabular}{@{}lcccccccccccc@{}}
\toprule
\multicolumn{13}{@{}l}{\textbf{Baseline Reference: Scenario 1 (Text-Only Black-Box)}} \\
\midrule
BERT & \multicolumn{12}{c}{\textbf{Accuracy:} 0.965 \quad | \quad \textbf{F1-Score:} 0.965 \quad | \quad \textbf{ROC-AUC:} 0.994} \\
\midrule
\midrule
\multirow{2}{*}{\textbf{Model Algorithm}} & \multicolumn{3}{c}{\textbf{Scenario 2: All\_40 (n=49)}} & \multicolumn{3}{c}{\textbf{Scen